# 1. V12 DINOv3-L LoRA material adaptation

Purpose: test one auditable representation change on official raw BDC Satria Data 2026 train images, fail cheaply on fold 0, and touch test only after a signed five-fold train-only adoption decision.

Kaggle prerequisites before Run All:

- latest Kaggle Python image;
- accelerator GPU T4 x2;
- internet enabled;
- the official competition dataset attached exactly once;
- accepted access to facebook/dinov3-vitl16-pretrain-lvd1689m;
- a private Kaggle secret named HF_TOKEN;
- replace TEAM_NAME below with the exact submission team name.

The parent notebook process performs CPU-only setup and download. All model construction and every CUDA call occur inside the two worker processes launched by accelerate.notebook_launcher.

## 2. Verified evidence from V7–V11

Repository evidence fixes the comparison, not the outcome. V7 achieved train-only OOF Macro F1 0.9901360067264733 with 287 errors and 241 Recyclable/Organic errors. The reproducible V10 control achieved 0.9901662323927614 with 287 and 242; fold 0 was 0.9897706675032462, 62 errors, 55 R/O errors, and Electronic F1 0.9955891619407687. V10 richer frozen pooling fell to 0.9882367633437932 on fold 0. V11 material-query attention was strongly rejected near 0.983106.

The official rules and project handoffs require visual-only official data, raw labels, a single model family, train-only selection, exact template order, and no leaderboard debugging. V5 proves the official Transformers ViT-L loading path. V7 proves that model_output.last_hidden_state is the official final-normalized sequence and that register tokens must be excluded from patch pooling. The V11 result directory is absent locally; its verified fold-0 result is therefore taken only from the task record, not invented.

## 3. Sol design decision

One recipe is locked:

- official Transformers DINOv3 ViT-L/16 at 320 by 320;
- runtime-discovered attention query and value nn.Linear targets only;
- PEFT LoRA rank 8, alpha 16, dropout 0.05;
- all pretrained weights frozen except LoRA and the official final LayerNorm affine parameters;
- final normalized CLS concatenated with the mean of final normalized true patch tokens;
- LayerNorm, Linear to 512, GELU, Dropout 0.30, main three-class output;
- one auxiliary binary R/O focal head from the same 512-dimensional feature, training only;
- weighted CE with label smoothing 0.05 plus 0.30 times balanced focal loss;
- controlled image augmentation, AdamW, eight epochs, patience three, and one seed 42;
- replicated two-process DDP with microbatch 2 per GPU and four accumulation steps.

The final LayerNorm affine is trainable because q/v adaptation changes the distribution entering that norm; its tiny parameter budget permits bounded recalibration while the exact official normalization operation is still applied once. Pretraining-time stochastic positional-coordinate shift, jitter, and rescale are disabled in the loaded config so the locked image transform is the only stochastic spatial augmentation; the 320px positional coordinates remain runtime-derived. ViT-B is rejected because the repository already establishes a stronger feasible ViT-L path. Resolution 288 sacrifices material detail; 384 increases activation and runtime risk. Key/output or all-linear LoRA enlarges the adaptation surface without direct evidence. Main-loss-only ignores the dominant R/O residual, while the auxiliary head is a regularizer and never routes inference. Full S2-FracMix is rejected because its saliency, fractal library, and manually tuned mixing controls add an unvalidated subsystem; only ordinary label-preserving transforms are retained. The official Transformers backend is chosen because V5 already proved its ViT-L model and processor contract.

Local evidence inspected: ref/papers/2508.10104v1.md (DINOv3), 2508.21041v3.md and 2603.28297v1.md (parameter-efficient adaptation), 2606.25784v1.md (S2-FracMix), and A_Review_of_Computer_Vision_for_Industrial-Grade_Waste_Classification.md. The supplied DINOv3 paper supports complementary normalized CLS and patch features. The supplied LoRA studies support low-rank q/v adaptation under domain shift, but do not prove this competition result. The industrial-waste review supports the material-versus-context failure hypothesis. These are motivation only; the locked gates decide adoption.

## 4. Locked competition, leakage, and adoption contract

Canonical supervision is the unchanged official raw train split: 9,999 Recyclable, 3,961 Electronic, and 12,567 Organic images. External images, pseudo-labels, manual label files, test-derived tuning, cleaning, blending, TTA, threshold shifts, row overrides, and fallback submissions are forbidden.

Fold 0 must clear every fixed gate before folds 1–4. A passing fold 0 creates a signed recipe lock. Five-fold OOF must then clear every fixed adoption gate before test discovery. A rejection writes diagnostics and a final status without searching for test or emitting any submission.

## 5. Minimal idempotent dependency setup

The latest compatible preinstalled torch, torchvision, transformers, safetensors, and huggingface_hub are reused. Only missing or too-old Accelerate/PEFT packages are installed. huggingface_hub is never pinned or downgraded. pip check is fatal.

In [1]:
# Must be set before importing torch/transformers/accelerate.
# PyTorch then uses NVML for CUDA availability checks, which avoids poisoning
# the fork used by Accelerate notebook_launcher on Kaggle.
import os
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"
os.environ["ACCELERATE_DEBUG_MODE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import importlib
import importlib.metadata as importlib_metadata
import re
import subprocess
import sys

def installed_version(distribution):
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return None

def major_minor(version):
    match = re.match(r"^(\d+)\.(\d+)", version or "")
    return tuple(map(int, match.groups())) if match else (-1, -1)

before_versions = {
    name: installed_version(name)
    for name in ("torch", "torchvision", "transformers", "accelerate", "peft",
                 "safetensors", "huggingface_hub", "torchao")
}
if major_minor(before_versions["transformers"]) < (4, 56):
    raise RuntimeError(
        "This notebook requires the latest Kaggle image with Transformers >= 4.56; "
        "it will not replace the preinstalled torch/transformers stack."
    )

# PEFT 0.19 probes optional TorchAO dispatchers whenever torchao is installed.
# Kaggle currently ships torchao 0.10.0, which PEFT rejects before ordinary
# non-quantized LoRA injection. V12 never uses TorchAO or quantization, so remove
# the incompatible optional package rather than upgrading the CUDA stack.
torchao_before = before_versions.get("torchao")
torchao_action = "not_installed"
if torchao_before is not None:
    uninstall = subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        text=True, capture_output=True, check=False,
    )
    if uninstall.returncode != 0:
        raise RuntimeError(
            "Could not remove optional incompatible torchao {}: {}".format(
                torchao_before, uninstall.stderr.strip()
            )
        )
    importlib.invalidate_caches()
    torchao_action = "uninstalled_optional_package_{}".format(torchao_before)

requirements = []
if major_minor(before_versions["accelerate"]) < (1, 10):
    requirements.append("accelerate>=1.10,<2")
if major_minor(before_versions["peft"]) < (0, 17):
    requirements.append("peft>=0.17,<1")
if requirements:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", *requirements,
    ])
    importlib.invalidate_caches()

after_versions = {
    name: installed_version(name)
    for name in ("torch", "torchvision", "transformers", "accelerate", "peft",
                 "safetensors", "huggingface_hub", "torchao")
}
if after_versions["torchao"] is not None:
    raise RuntimeError(
        "Optional torchao remained installed after cleanup: {}".format(
            after_versions["torchao"]
        )
    )
if major_minor(after_versions["accelerate"]) < (1, 10) or major_minor(after_versions["peft"]) < (0, 17):
    raise RuntimeError("Accelerate/PEFT minimum versions were not satisfied.")

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    text=True, capture_output=True, check=False,
)

# Kaggle's base image can contain unrelated optional-package inconsistencies
# (for example BigFrames/Colab/Dopamine/MoviePy). Those packages are not
# imported by this notebook and must not abort a long GPU experiment.
# Fail closed only when pip reports a conflict involving the V12 runtime stack.
pip_check_lines = [
    line.strip()
    for line in (pip_check.stdout + "\n" + pip_check.stderr).splitlines()
    if line.strip()
]
critical_dependency_markers = (
    "torch ",
    "torchvision ",
    "transformers ",
    "accelerate ",
    "peft ",
    "safetensors ",
    "huggingface-hub ",
    "huggingface_hub ",
)
critical_pip_conflicts = [
    line
    for line in pip_check_lines
    if any(marker in line.lower() for marker in critical_dependency_markers)
]
noncritical_pip_warnings = [
    line for line in pip_check_lines if line not in critical_pip_conflicts
]

DEPENDENCY_SETUP = {
    "before": before_versions,
    "requested": requirements,
    "torchao_action": torchao_action,
    "after": after_versions,
    "pip_check_returncode": pip_check.returncode,
    "pip_check_stdout": pip_check.stdout.strip(),
    "pip_check_stderr": pip_check.stderr.strip(),
    "critical_pip_conflicts": critical_pip_conflicts,
    "noncritical_pip_warnings": noncritical_pip_warnings,
    "decision": (
        "fail_on_v12_runtime_conflict"
        if critical_pip_conflicts
        else "continue_noncritical_kaggle_base_image_warnings"
    ),
}
print(DEPENDENCY_SETUP)

if critical_pip_conflicts:
    raise RuntimeError(
        "pip check found conflicts in the V12 runtime stack: {}".format(
            critical_pip_conflicts
        )
    )


{'before': {'torch': '2.10.0+cu128', 'torchvision': '0.25.0+cu128', 'transformers': '5.0.0', 'accelerate': '1.13.0', 'peft': '0.19.1', 'safetensors': '0.7.0', 'huggingface_hub': '1.11.0', 'torchao': '0.10.0'}, 'requested': [], 'torchao_action': 'uninstalled_optional_package_0.10.0', 'after': {'torch': '2.10.0+cu128', 'torchvision': '0.25.0+cu128', 'transformers': '5.0.0', 'accelerate': '1.13.0', 'peft': '0.19.1', 'safetensors': '0.7.0', 'huggingface_hub': '1.11.0', 'torchao': None}, 'pip_check_returncode': 1, 'pip_check_stdout': 'bigframes 2.39.0 requires google-cloud-bigquery-storage, which is not installed.\ngoogle-adk 1.29.0 requires google-cloud-bigquery-storage, which is not installed.\ngoogle-colab 1.0.0 has requirement jupyter-server==2.14.0, but you have jupyter-server 2.12.5.\ngoogle-colab 1.0.0 has requirement pandas==2.2.2, but you have pandas 2.3.3.\ndopamine-rl 4.1.2 has requirement gym<=0.25.2, but you have gym 0.26.2.\nmoviepy 1.0.3 has requirement decorator<5.0,>=4.0.2,

## 6. Imports and immutable configuration

This cell imports frameworks but makes no CUDA/device call. Accelerator construction, model loading, GPU inspection, and seeding remain inside the launched worker.

In [2]:
import contextlib
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
import zipfile
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ACCELERATE_DEBUG_MODE"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import accelerate
import matplotlib
import numpy as np
import pandas as pd
import peft
import PIL
import safetensors
import sklearn
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import transformers
transformers.utils.logging.disable_progress_bar()
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass
from accelerate import Accelerator, notebook_launcher
from accelerate.utils import DistributedDataParallelKwargs, set_seed
from huggingface_hub import model_info, snapshot_download
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset, DistributedSampler, Sampler
from torchvision.transforms import InterpolationMode, v2
from transformers import AutoConfig, AutoImageProcessor, AutoModel, get_cosine_schedule_with_warmup

TEAM_NAME = "SD2026040000412"
if TEAM_NAME == "CHANGE_ME":
    raise ValueError("Set TEAM_NAME before Kaggle Run All; the modeling recipe is unaffected.")

EXPERIMENT_NAME = "v12_dinov3_vitl_lora_material_adaptation"
MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
MODEL_REVISION_REQUEST = "main"
IMAGE_SIZE = 320
NUM_CLASSES = 3
CLASS_NAMES = ("Recyclable", "Electronic", "Organic")
CLASS_DIR_TO_LABEL = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
EXPECTED_RAW_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_TRAIN_ROWS = 26527
EXPECTED_TEST_ROWS = 1458
SUBMISSION_COLUMNS = ("id", "predicted")
ALLOWED_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

SEED = 42
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
ASSESSMENT_FOLDS = (1, 2, 3, 4)
EXPECTED_FOLD_SIGNATURE = "0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c"

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TRAIN_FINAL_NORM = True
HEAD_HIDDEN = 512
HEAD_DROPOUT = 0.30
LABEL_SMOOTHING = 0.05
AUXILIARY_WEIGHT = 0.30
FOCAL_GAMMA = 2.0
LORA_LR = 1e-4
NORM_HEAD_LR = 3e-4
WEIGHT_DECAY = 0.075
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0
MAX_EPOCHS = 8
EARLY_STOPPING_PATIENCE = 3
MICROBATCH_PER_GPU = 2
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH = 2 * MICROBATCH_PER_GPU * GRAD_ACCUM_STEPS
LOADER_WORKERS_PER_PROCESS = 2
RELOAD_TOLERANCE = 1e-6

V10_FOLD_REFERENCE = {
    0: {"macro_f1": 0.9897706675032462, "total_errors": 62, "ro_errors": 55,
        "f1_electronic": 0.9955891619407687},
    1: {"macro_f1": 0.9903469894805695, "total_errors": 56, "ro_errors": 47,
        "f1_electronic": 0.994343180389692},
    2: {"macro_f1": 0.9908291600511604, "total_errors": 53, "ro_errors": 44,
        "f1_electronic": 0.9943145925457991},
    3: {"macro_f1": 0.9890345290445208, "total_errors": 64, "ro_errors": 54,
        "f1_electronic": 0.9936868686868687},
    4: {"macro_f1": 0.9908498966736979, "total_errors": 52, "ro_errors": 42,
        "f1_electronic": 0.9937106918238994},
}
V10_OOF_REFERENCE = {
    "macro_f1": 0.9901662323927614,
    "total_errors": 287,
    "ro_errors": 242,
    "f1_electronic": 0.994328922495274,
}
DEVELOPMENT_THRESHOLDS = {
    "min_macro_f1": 0.9910,
    "min_gain_over_v10_fold0": 0.0010,
    "max_total_errors": 57,
    "max_ro_errors": 50,
    "max_electronic_f1_decline": 0.0005,
}
ADOPTION_THRESHOLDS = {
    "min_oof_macro_f1": 0.9910,
    "max_oof_errors": 280,
    "max_oof_ro_errors": 235,
    "min_fold_wins": 3,
    "max_electronic_f1_decline": 0.0005,
    "max_single_positive_gain_share": 0.75,
}

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
if not KAGGLE_INPUT.is_dir() or not KAGGLE_WORKING.is_dir():
    raise RuntimeError("Run on Kaggle with the official dataset attached.")
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
FOLDS_DIR = ARTIFACT_DIR / "folds"
FIGURES_DIR = ARTIFACT_DIR / "figures"
for directory in (ARTIFACT_DIR, FOLDS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

Image.MAX_IMAGE_PIXELS = 200_000_000
ImageFile.LOAD_TRUNCATED_IMAGES = False
NOTEBOOK_STARTED_AT = time.perf_counter()
TEST_ACCESS_STATE = {
    "unlocked": False,
    "test_images_accessed": False,
    "test_template_accessed": False,
    "external_or_manual_labels_used": False,
}


## 7. Determinism, hashing, metrics, logging, and artifact helpers

All writes before launch are single-parent CPU writes. During distributed execution, callers enforce accelerator.is_main_process before every mutation. Probability metrics use explicit row normalization before log loss.

In [3]:
def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError("Not JSON serializable: {}".format(type(value).__name__))

def sha256_json(value):
    payload = json.dumps(
        value, sort_keys=True, separators=(",", ":"), ensure_ascii=False,
        allow_nan=False, default=json_default,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                return digest.hexdigest()
            digest.update(chunk)

def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True, allow_nan=False, default=json_default)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)

def write_npy_atomic(array, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp.npy")
    np.save(temporary, np.asarray(array), allow_pickle=False)
    os.replace(temporary, path)

def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def signed_document(payload, signature_field="signature"):
    if signature_field in payload:
        raise ValueError("Payload already contains its signature field.")
    document = dict(payload)
    document[signature_field] = sha256_json(payload)
    return document

def verify_signed_document(document, signature_field="signature"):
    unsigned = dict(document)
    signature = unsigned.pop(signature_field, None)
    return bool(signature and signature == sha256_json(unsigned))

def normalize_probabilities(probabilities):
    probs = np.asarray(probabilities, dtype=np.float64)
    if probs.ndim != 2 or probs.shape[1] != NUM_CLASSES or not np.isfinite(probs).all():
        raise RuntimeError("Probability tensor is nonfinite or has the wrong shape.")
    if (probs < 0).any():
        raise RuntimeError("Probability tensor contains negative values.")
    probs = probs / probs.sum(axis=1, keepdims=True).clip(min=1e-12)
    if not np.isfinite(probs).all() or not np.allclose(probs.sum(axis=1), 1.0, atol=1e-7):
        raise RuntimeError("Probability normalization failed.")
    return probs.astype(np.float32)

def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probs = normalize_probabilities(probabilities)
    predictions = probs.argmax(axis=1).astype(np.int64)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1, 2])
    per_class_f1 = f1_score(
        labels, predictions, labels=[0, 1, 2], average=None, zero_division=0
    )
    errors = predictions != labels
    ro_mask = ((labels == 0) & (predictions == 2)) | ((labels == 2) & (predictions == 0))
    return {
        "rows": int(len(labels)),
        "macro_f1": float(f1_score(
            labels, predictions, labels=[0, 1, 2], average="macro", zero_division=0
        )),
        "accuracy": float((~errors).mean()),
        "log_loss": float(log_loss(labels, probs, labels=[0, 1, 2])),
        "total_errors": int(errors.sum()),
        "ro_errors": int(ro_mask.sum()),
        "recyclable_to_organic": int(((labels == 0) & (predictions == 2)).sum()),
        "organic_to_recyclable": int(((labels == 2) & (predictions == 0)).sum()),
        "f1_recyclable": float(per_class_f1[0]),
        "f1_electronic": float(per_class_f1[1]),
        "f1_organic": float(per_class_f1[2]),
        "confusion_matrix": matrix.tolist(),
        "probabilities_finite": True,
        "probability_rows_normalized": True,
    }

def metric_rank(metrics):
    return (
        metrics["macro_f1"],
        -metrics["total_errors"],
        -metrics["ro_errors"],
        -metrics["log_loss"],
    )

def broadcast_object(accelerator, value):
    objects = [value if accelerator.is_main_process else None]
    dist.broadcast_object_list(objects, src=0)
    return objects[0]

def cpu_state_dict(module):
    return {name: tensor.detach().cpu() for name, tensor in module.state_dict().items()}

def safe_team_name(value):
    cleaned = "".join(
        character if character.isalnum() or character in {"-", "_"} else "_"
        for character in str(value)
    ).strip("_")
    if not cleaned:
        raise ValueError("TEAM_NAME has no safe filename characters.")
    return cleaned

def plot_confusion(metrics, path, title):
    import matplotlib.pyplot as plt
    matrix = np.asarray(metrics["confusion_matrix"], dtype=np.int64)
    figure, axis = plt.subplots(figsize=(4.8, 4.2))
    image = axis.imshow(matrix, cmap="Blues")
    for row in range(3):
        for column in range(3):
            axis.text(column, row, str(matrix[row, column]), ha="center", va="center")
    axis.set_xticks(range(3), CLASS_NAMES, rotation=25, ha="right")
    axis.set_yticks(range(3), CLASS_NAMES)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    axis.set_title(title)
    figure.colorbar(image, ax=axis, fraction=0.046)
    figure.tight_layout()
    temporary = Path(path).with_name(Path(path).name + ".tmp.png")
    figure.savefig(temporary, dpi=150)
    plt.close(figure)
    os.replace(temporary, path)

def build_artifact_bundle():
    bundle = ARTIFACT_DIR / "artifact_bundle.zip"
    temporary = bundle.with_name(bundle.name + ".tmp")
    forbidden_parts = {"pretrained_snapshot", "hf_cache", "optimizer_state", "feature_cache"}
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(ARTIFACT_DIR.rglob("*")):
            if not path.is_file() or path in {bundle, temporary} or ".tmp" in path.name:
                continue
            relative = path.relative_to(ARTIFACT_DIR)
            lowered = {part.lower() for part in relative.parts}
            if lowered & forbidden_parts:
                continue
            archive.write(path, relative.as_posix())
    os.replace(temporary, bundle)
    with zipfile.ZipFile(bundle, "r") as archive:
        names = archive.namelist()
        if any(set(Path(name).parts) & forbidden_parts for name in names):
            raise RuntimeError("Artifact bundle contains a forbidden large/transient path.")
    return bundle

DEPENDENCY_AUDIT = signed_document({
    "strategy": "reuse Kaggle torch/torchvision/transformers; install only missing/old accelerate or peft",
    "huggingface_hub_pinned_or_downgraded": False,
    **DEPENDENCY_SETUP,
})
write_json_atomic(DEPENDENCY_AUDIT, ARTIFACT_DIR / "dependency_audit.json")

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        name: importlib_metadata.version(name)
        for name in (
            "torch", "torchvision", "transformers", "accelerate", "peft",
            "safetensors", "huggingface_hub", "scikit-learn", "numpy", "pandas", "Pillow"
        )
    },
    "cuda_runtime": None,
    "cudnn": None,
    "gpus": None,
    "gpu_fields_populated_inside_workers": True,
}
write_json_atomic(ENVIRONMENT, ARTIFACT_DIR / "environment.json")


## 8. Train-only official data discovery and validation

Discovery prunes every directory named test and never opens a test image or template. It accepts exactly one train root with the official class directories/counts and rejects the known manual-label input name.

In [4]:
def walk_without_test(base, max_depth=4):
    base = Path(base)
    for current, directories, files in os.walk(base):
        current_path = Path(current)
        depth = len(current_path.relative_to(base).parts)
        directories[:] = [
            name for name in directories
            if name.lower() != "test" and not name.startswith(".") and depth < max_depth
        ]
        yield current_path, tuple(directories), tuple(files)

def exact_train_counts(path):
    path = Path(path)
    try:
        children = list(path.iterdir())
        if {child.name for child in children} != set(CLASS_DIR_TO_LABEL):
            return None
        if any(child.is_symlink() or not child.is_dir() for child in children):
            return None
        counts = {}
        for directory_name, label in CLASS_DIR_TO_LABEL.items():
            entries = list((path / directory_name).iterdir())
            if any(
                entry.is_symlink() or not entry.is_file()
                or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
                for entry in entries
            ):
                return None
            counts[label] = len(entries)
        return counts
    except OSError:
        return None

def discover_official_train():
    candidates = []
    forbidden = []
    for current, _, files in walk_without_test(KAGGLE_INPUT):
        if current.name.lower() == "train":
            candidates.append(current)
        for filename in files:
            if filename.lower() == "manual_labeling.csv":
                forbidden.append(current / filename)
    if forbidden:
        raise RuntimeError("Detach manual_labeling.csv; V12 uses raw official labels only.")
    matches = {
        candidate.resolve()
        for candidate in candidates
        if exact_train_counts(candidate) == EXPECTED_RAW_COUNTS
    }
    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one official raw train root, found {}: {}".format(
                len(matches), sorted(map(str, matches))
            )
        )
    return matches.pop()

def index_official_train(train_root):
    rows = []
    for directory_name, label in CLASS_DIR_TO_LABEL.items():
        directory = train_root / directory_name
        for path in sorted(directory.iterdir(), key=lambda item: item.name):
            rows.append({
                "image_id": path.relative_to(train_root).as_posix(),
                "path": str(path.resolve()),
                "label": int(label),
                "class_name": CLASS_NAMES[label],
                "file_size_bytes": int(path.stat().st_size),
            })
    frame = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)
    checks = {
        "rows": len(frame) == EXPECTED_TRAIN_ROWS,
        "unique_ids": frame["image_id"].is_unique,
        "raw_counts": frame["label"].value_counts().sort_index().to_dict() == EXPECTED_RAW_COUNTS,
        "labels": set(frame["label"]) == {0, 1, 2},
    }
    if not all(checks.values()):
        raise RuntimeError("Official train contract failed: {}".format(checks))
    return frame

TRAIN_ROOT = discover_official_train()
train_df = index_official_train(TRAIN_ROOT)
print("Official raw train:", TRAIN_ROOT, "| rows:", len(train_df))


Official raw train: /kaggle/input/datasets/riantamaputra/bdc-2026-waste-classification-dataset/train | rows: 26527


## 9. Exact fold reconstruction and signature validation

Rows are sorted by canonical relative image ID before the V10 split is recomputed. Any mismatch with the persisted V10 SHA-256 is fatal.

In [5]:
train_df["fold"] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, validation_indices) in enumerate(splitter.split(train_df, train_df["label"])):
    train_df.loc[validation_indices, "fold"] = fold
train_df["fold"] = train_df["fold"].astype(np.int64)

FOLD_ASSIGNMENT_SIGNATURE = sha256_json(
    train_df[["image_id", "label", "fold"]].to_dict("records")
)
if FOLD_ASSIGNMENT_SIGNATURE != EXPECTED_FOLD_SIGNATURE:
    raise RuntimeError(
        "Stable V10 fold signature mismatch: {} != {}".format(
            FOLD_ASSIGNMENT_SIGNATURE, EXPECTED_FOLD_SIGNATURE
        )
    )

FOLD_SIGNATURE = signed_document({
    "implementation": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "canonical_order": "relative image_id ascending",
    "development_fold": DEVELOPMENT_FOLD,
    "assessment_folds": list(ASSESSMENT_FOLDS),
    "fold_assignment_sha256": FOLD_ASSIGNMENT_SIGNATURE,
    "class_counts_by_fold": train_df.groupby(["fold", "label"]).size().unstack(fill_value=0).to_dict("index"),
})
DATA_CONTRACT = {
    "canonical_source": "official raw train only",
    "train_root": str(TRAIN_ROOT),
    "rows": len(train_df),
    "raw_counts": EXPECTED_RAW_COUNTS,
    "class_mapping": {str(value): name for value, name in enumerate(CLASS_NAMES)},
    "ordered_ids_sha256": sha256_json(train_df["image_id"].tolist()),
    "labels_sha256": sha256_json(train_df["label"].astype(int).tolist()),
    "source_identity_sha256": sha256_json(
        train_df[["image_id", "label", "file_size_bytes"]].to_dict("records")
    ),
    "fold_assignment_sha256": FOLD_ASSIGNMENT_SIGNATURE,
    "dropped_rows": 0,
    "relabeled_rows": 0,
    "external_images_or_labels": False,
    "test_images_accessed": False,
}
DATA_CONTRACT["data_contract_signature"] = sha256_json(DATA_CONTRACT)

write_csv_atomic(
    train_df[["image_id", "label", "class_name", "fold"]],
    ARTIFACT_DIR / "fold_assignments.csv",
)
write_json_atomic(FOLD_SIGNATURE, ARTIFACT_DIR / "fold_signature.json")
write_json_atomic(DATA_CONTRACT, ARTIFACT_DIR / "data_contract.json")
print("Verified fold signature:", FOLD_ASSIGNMENT_SIGNATURE)


Verified fold signature: 0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c


## 10. CPU-only immutable model revision resolution and download

The gated revision and snapshot are resolved before launch without constructing a model or touching CUDA. Downloaded pretrained weights remain outside the artifact directory and are excluded from the ZIP.

In [6]:
def resolve_hf_token():
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            "Accept the DINOv3 model license and add a private Kaggle secret named HF_TOKEN."
        )
    return token

_hf_token = resolve_hf_token()
try:
    RESOLVED_MODEL_REVISION = model_info(
        MODEL_ID, revision=MODEL_REVISION_REQUEST, token=_hf_token
    ).sha
    if (
        len(RESOLVED_MODEL_REVISION or "") != 40
        or any(character not in "0123456789abcdef" for character in RESOLVED_MODEL_REVISION.lower())
    ):
        raise RuntimeError("The Hub did not return an immutable 40-character revision.")
    MODEL_SNAPSHOT_DIR = Path(snapshot_download(
        repo_id=MODEL_ID,
        revision=RESOLVED_MODEL_REVISION,
        token=_hf_token,
        allow_patterns=[
            "*.json", "*.safetensors", "*.safetensors.index.json",
            "preprocessor_config.json",
        ],
    ))
finally:
    del _hf_token

BASE_CONFIG = {
    "experiment": EXPERIMENT_NAME,
    "team_name": TEAM_NAME,
    "model_id": MODEL_ID,
    "model_revision": RESOLVED_MODEL_REVISION,
    "backend": "official Transformers AutoModel",
    "image_size": IMAGE_SIZE,
    "seed": SEED,
    "fold_signature": FOLD_ASSIGNMENT_SIGNATURE,
    "lora": {"targets": "runtime-audited query/value Linear modules", "r": LORA_RANK,
             "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "readout": "official final normalized CLS + mean true patches -> LN -> Linear(512) -> GELU -> Dropout(0.30)",
    "train_final_norm": TRAIN_FINAL_NORM,
    "loss": {"main": "partition-weighted CE", "label_smoothing": LABEL_SMOOTHING,
             "auxiliary": "balanced R/O binary focal", "auxiliary_weight": AUXILIARY_WEIGHT,
             "focal_gamma": FOCAL_GAMMA},
    "augmentation": {
        "random_resized_crop": {"size": IMAGE_SIZE, "scale": [0.65, 1.0], "ratio": [0.8, 1.25]},
        "horizontal_flip_probability": 0.5,
        "rotation_degrees": 12,
        "color_jitter": [0.15, 0.15, 0.10, 0.02],
        "grayscale_probability": 0.10,
        "gaussian_blur_probability": 0.10,
        "random_erasing_probability": 0.15,
        "mixup_or_cutmix": False,
        "validation_and_test": "one deterministic direct-square original view",
    },
    "optimization": {
        "optimizer": "AdamW", "lora_lr": LORA_LR, "norm_head_lr": NORM_HEAD_LR,
        "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
        "scheduler": "cosine", "gradient_clip": GRAD_CLIP,
        "max_epochs": MAX_EPOCHS, "patience": EARLY_STOPPING_PATIENCE,
        "microbatch_per_gpu": MICROBATCH_PER_GPU,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "effective_batch": EFFECTIVE_BATCH,
    },
    "development_thresholds": DEVELOPMENT_THRESHOLDS,
    "adoption_thresholds": ADOPTION_THRESHOLDS,
    "test_policy": {
        "locked_until_signed_adoption": True, "views": 1, "tta": False,
        "fold_probability_average": 5, "external_blend": False,
        "threshold_shift": False, "row_override": False,
    },
}
CONFIG_SIGNATURE = sha256_json({key: value for key, value in BASE_CONFIG.items() if key != "team_name"})
BASE_CONFIG["modeling_config_signature"] = CONFIG_SIGNATURE
write_json_atomic(BASE_CONFIG, ARTIFACT_DIR / "config.json")
print("Pinned model revision:", RESOLVED_MODEL_REVISION)
print("CPU-only snapshot ready:", MODEL_SNAPSHOT_DIR)


Pinned model revision: ea8dc2863c51be0a264bab82070e3e8836b02d51
CPU-only snapshot ready: /root/.cache/huggingface/hub/models--facebook--dinov3-vitl16-pretrain-lvd1689m/snapshots/ea8dc2863c51be0a264bab82070e3e8836b02d51


## 11. Distributed-safe model, LoRA, dataset, loss, and optimizer definitions

The worker enumerates actual named modules, requires exactly one query and one value Linear per runtime transformer layer, persists all full target names before injection, and then applies one PEFT path. The final sequence always comes from last_hidden_state; no manual second normalization is applied.

In [7]:
def resolve_module(root, dotted_name):
    module = root
    for component in dotted_name.split("."):
        module = getattr(module, component)
    return module

def discover_final_norm(model, hidden_size):
    candidates = []
    for name, module in model.named_modules():
        if not name or not isinstance(module, nn.LayerNorm):
            continue
        numeric_scope = any(component.isdigit() for component in name.split("."))
        normalized_shape = tuple(int(value) for value in module.normalized_shape)
        if not numeric_scope and normalized_shape == (hidden_size,):
            candidates.append((name, module))
    if len(candidates) != 1:
        raise RuntimeError(
            "Could not uniquely discover the official top-level final LayerNorm: {}".format(
                [name for name, _ in candidates]
            )
        )
    return candidates[0]

def discover_lora_targets(model, hidden_size, num_hidden_layers):
    # Names are inspected first; aliases cover the official pre-5 and 5.x implementations.
    role_aliases = {
        "query": "query", "q_proj": "query",
        "value": "value", "v_proj": "value",
    }
    records = []
    for full_name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        components = full_name.split(".")
        lowered = [component.lower() for component in components]
        if not any("attention" in component or component == "attn" for component in lowered):
            continue
        role = role_aliases.get(lowered[-1])
        if role is None:
            continue
        numeric_components = [int(component) for component in components if component.isdigit()]
        if len(numeric_components) != 1:
            raise RuntimeError("Ambiguous transformer block index in target {}".format(full_name))
        records.append({
            "full_name": full_name,
            "role": role,
            "block": numeric_components[0],
            "module_class": type(module).__name__,
            "in_features": int(module.in_features),
            "out_features": int(module.out_features),
        })
    by_block = {}
    for record in records:
        by_block.setdefault(record["block"], []).append(record)
    expected_blocks = set(range(num_hidden_layers))
    checks = {
        "block_ids_exact": set(by_block) == expected_blocks,
        "two_targets_per_block": all(len(by_block.get(block, [])) == 2 for block in expected_blocks),
        "query_and_value_per_block": all(
            {item["role"] for item in by_block.get(block, [])} == {"query", "value"}
            for block in expected_blocks
        ),
        "linear_hidden_to_hidden": all(
            item["module_class"] == "Linear"
            and item["in_features"] == hidden_size
            and item["out_features"] == hidden_size
            for item in records
        ),
        "unique_full_names": len({item["full_name"] for item in records}) == 2 * num_hidden_layers,
    }
    if not all(checks.values()):
        raise RuntimeError("LoRA target discovery failed closed: {} | {}".format(checks, records))
    records.sort(key=lambda item: (item["block"], item["role"]))
    return records, checks

def processor_contract(processor):
    mean = tuple(float(value) for value in getattr(processor, "image_mean", ()))
    std = tuple(float(value) for value in getattr(processor, "image_std", ()))
    factor = float(getattr(processor, "rescale_factor", float("nan")))
    try:
        resampling = Image.Resampling(int(processor.resample))
    except Exception as exc:
        raise RuntimeError("Unsupported official processor resampling mode.") from exc
    checks = {
        "resize_enabled": getattr(processor, "do_resize", None) is True,
        "rescale_is_1_over_255": (
            getattr(processor, "do_rescale", None) is True and np.isclose(factor, 1.0 / 255.0)
        ),
        "normalization_valid": (
            getattr(processor, "do_normalize", None) is True
            and len(mean) == 3 and len(std) == 3
            and np.isfinite(mean).all() and np.isfinite(std).all() and min(std) > 0
        ),
        "no_center_crop": getattr(processor, "do_center_crop", None) is not True,
        "rgb_compatible": getattr(processor, "do_convert_rgb", None) in (None, True),
    }
    if not all(checks.values()):
        raise RuntimeError("Official processor contract failed: {}".format(checks))
    return {
        "processor_class": type(processor).__name__,
        "mean": list(mean),
        "std": list(std),
        "rescale_factor": factor,
        "resample_name": resampling.name,
        "spatial_override": [IMAGE_SIZE, IMAGE_SIZE],
        "direct_square": True,
        "checks": checks,
    }

class V12Classifier(nn.Module):
    def __init__(self, adapted_backbone, hidden_size, num_register_tokens, patch_count, final_norm_path):
        super().__init__()
        self.backbone = adapted_backbone
        self.hidden_size = int(hidden_size)
        self.num_register_tokens = int(num_register_tokens)
        self.patch_count = int(patch_count)
        self.final_norm_path = final_norm_path
        self.features = nn.Sequential(
            nn.LayerNorm(2 * hidden_size),
            nn.Linear(2 * hidden_size, HEAD_HIDDEN),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
        )
        self.main_classifier = nn.Linear(HEAD_HIDDEN, NUM_CLASSES)
        self.auxiliary_classifier = nn.Linear(HEAD_HIDDEN, 1)

    def forward(self, pixel_values, return_tokens=False):
        output = self.backbone(pixel_values=pixel_values, return_dict=True)
        tokens = output.last_hidden_state
        patch_start = 1 + self.num_register_tokens
        expected_sequence = patch_start + self.patch_count
        if (
            tokens.ndim != 3
            or tokens.shape[1] != expected_sequence
            or tokens.shape[2] != self.hidden_size
        ):
            raise RuntimeError(
                "Runtime final-token contract failed: shape={} expected=({}, {}, {}).".format(
                    tuple(tokens.shape), pixel_values.shape[0], expected_sequence, self.hidden_size
                )
            )
        cls_token = tokens[:, 0, :]
        true_patch_tokens = tokens[:, patch_start:, :]
        if true_patch_tokens.shape[1] != self.patch_count:
            raise RuntimeError("Register exclusion did not yield the runtime-derived patch count.")
        pooled = torch.cat((cls_token, true_patch_tokens.mean(dim=1)), dim=1)
        features = self.features(pooled)
        result = {
            "main_logits": self.main_classifier(features),
            "auxiliary_logits": self.auxiliary_classifier(features).squeeze(1),
        }
        if return_tokens:
            result["tokens"] = tokens
            result["true_patch_tokens"] = true_patch_tokens
        return result

def build_runtime_model(accelerator):
    processor = AutoImageProcessor.from_pretrained(
        MODEL_SNAPSHOT_DIR, local_files_only=True
    )
    preprocess = processor_contract(processor)
    model_config = AutoConfig.from_pretrained(MODEL_SNAPSHOT_DIR, local_files_only=True)
    original_position_augmentation = {
        name: getattr(model_config, name, None)
        for name in ("pos_embed_shift", "pos_embed_jitter", "pos_embed_rescale")
    }
    for name in original_position_augmentation:
        if hasattr(model_config, name):
            setattr(model_config, name, None)

    base = AutoModel.from_pretrained(
        MODEL_SNAPSHOT_DIR,
        config=model_config,
        local_files_only=True,
        dtype=torch.float32,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    if getattr(base.config, "_commit_hash", RESOLVED_MODEL_REVISION) not in (
        None, RESOLVED_MODEL_REVISION
    ):
        raise RuntimeError("Loaded model revision differs from the pinned snapshot.")

    hidden_size = int(base.config.hidden_size)
    num_hidden_layers = int(base.config.num_hidden_layers)
    num_register_tokens = int(base.config.num_register_tokens)
    patch_size = base.config.patch_size
    if isinstance(patch_size, (list, tuple)):
        if len(patch_size) != 2 or patch_size[0] != patch_size[1]:
            raise RuntimeError("Only square patches are supported.")
        patch_size = patch_size[0]
    patch_size = int(patch_size)
    if patch_size != 16 or IMAGE_SIZE % patch_size:
        raise RuntimeError("Pinned ViT-L/16 and 320px contract failed.")
    patch_grid = IMAGE_SIZE // patch_size
    patch_count = patch_grid * patch_grid
    if patch_grid != 20 or patch_count != 400:
        raise RuntimeError("320px ViT-L/16 must derive a 20 by 20 patch grid.")
    if int(getattr(base.config, "num_channels", 0)) != 3:
        raise RuntimeError("Loaded backbone is not RGB.")

    final_norm_path, final_norm = discover_final_norm(base, hidden_size)
    target_records, target_checks = discover_lora_targets(
        base, hidden_size, num_hidden_layers
    )
    target_names = [record["full_name"] for record in target_records]
    target_audit = {
        "model_id": MODEL_ID,
        "model_revision": RESOLVED_MODEL_REVISION,
        "runtime_num_hidden_layers": num_hidden_layers,
        "expected_target_count": 2 * num_hidden_layers,
        "actual_target_count": len(target_names),
        "targets": target_records,
        "checks": target_checks,
        "persisted_before_injection": True,
    }
    target_audit["target_signature"] = sha256_json(target_audit)
    gathered_signatures = [None] * accelerator.num_processes
    dist.all_gather_object(gathered_signatures, target_audit["target_signature"])
    if len(set(gathered_signatures)) != 1:
        raise RuntimeError("Workers discovered different LoRA targets.")
    if accelerator.is_main_process:
        audit_path = ARTIFACT_DIR / "lora_target_audit.json"
        if audit_path.is_file():
            existing = read_json(audit_path)
            if existing.get("target_signature") != target_audit["target_signature"]:
                raise RuntimeError("Existing LoRA target audit is incompatible.")
        write_json_atomic(target_audit, audit_path)
    accelerator.wait_for_everyone()

    base.requires_grad_(False)
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_names,
        bias="none",
        inference_mode=False,
    )
    adapted = get_peft_model(base, lora_config)
    underlying = adapted.get_base_model()
    final_norm = resolve_module(underlying, final_norm_path)
    for parameter in final_norm.parameters():
        parameter.requires_grad_(TRAIN_FINAL_NORM)

    checkpointing_supported = hasattr(adapted, "gradient_checkpointing_enable")
    checkpointing_enabled = False
    if checkpointing_supported:
        try:
            adapted.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        except TypeError:
            adapted.gradient_checkpointing_enable()
        if hasattr(adapted, "enable_input_require_grads"):
            adapted.enable_input_require_grads()
        checkpointing_enabled = True

    model = V12Classifier(
        adapted, hidden_size, num_register_tokens, patch_count, final_norm_path
    )
    final_norm_parameter_ids = {id(parameter) for parameter in final_norm.parameters()}
    unintended_backbone_trainables = []
    lora_trainables = []
    final_norm_trainables = []
    for name, parameter in adapted.named_parameters():
        if not parameter.requires_grad:
            continue
        if "lora_" in name:
            lora_trainables.append(name)
        elif id(parameter) in final_norm_parameter_ids:
            final_norm_trainables.append(name)
        else:
            unintended_backbone_trainables.append(name)
    if unintended_backbone_trainables:
        raise RuntimeError(
            "Unintended pretrained backbone weights are trainable: {}".format(
                unintended_backbone_trainables
            )
        )
    if not lora_trainables or len(final_norm_trainables) != 2:
        raise RuntimeError("LoRA/final-norm trainable audit failed.")

    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    lora_parameters = sum(
        parameter.numel()
        for name, parameter in adapted.named_parameters()
        if parameter.requires_grad and "lora_" in name
    )
    expected_lora_tensor_count = 2 * len(target_names)
    expected_lora_parameters = (
        len(target_names) * 2 * hidden_size * LORA_RANK
    )
    exact_lora_injection_checks = {
        "two_trainable_lora_tensors_per_target": (
            len(lora_trainables) == expected_lora_tensor_count
        ),
        "exact_lora_parameter_count": (
            lora_parameters == expected_lora_parameters
        ),
    }
    if not all(exact_lora_injection_checks.values()):
        raise RuntimeError(
            "PEFT did not inject the exact audited Q/V LoRA recipe: {} | "
            "trainable_lora_tensors={} expected={} | lora_parameters={} expected={}".format(
                exact_lora_injection_checks,
                len(lora_trainables),
                expected_lora_tensor_count,
                lora_parameters,
                expected_lora_parameters,
            )
        )
    if not (500_000 <= trainable_parameters <= 5_000_000):
        raise RuntimeError(
            "Implausible trainable parameter count: {:,}".format(trainable_parameters)
        )
    if trainable_parameters / total_parameters >= 0.02:
        raise RuntimeError("Trainable fraction is too large for the locked PEFT recipe.")

    runtime_recipe = {
        "base_config_signature": CONFIG_SIGNATURE,
        "model_id": MODEL_ID,
        "model_revision": RESOLVED_MODEL_REVISION,
        "backend": "Transformers plus PEFT",
        "attention_implementation": "sdpa",
        "determinism": {
            "cudnn_benchmark": False,
            "cudnn_deterministic": True,
            "deterministic_algorithms": "warn_only",
        },
        "image_size": IMAGE_SIZE,
        "preprocess": preprocess,
        "pretraining_position_augmentation_original": original_position_augmentation,
        "pretraining_position_augmentation_during_v12": None,
        "hidden_size": hidden_size,
        "num_hidden_layers": num_hidden_layers,
        "patch_size": patch_size,
        "patch_grid": patch_grid,
        "true_patch_count": patch_count,
        "num_register_tokens": num_register_tokens,
        "sequence_count_formula": "1 + runtime registers + runtime patches",
        "official_final_norm_path": final_norm_path,
        "official_final_norm_trainable": TRAIN_FINAL_NORM,
        "official_output_source": "model_output.last_hidden_state",
        "lora_target_signature": target_audit["target_signature"],
        "lora_target_names": target_names,
        "lora": {
            "rank": LORA_RANK,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_count": len(target_names),
            "expected_trainable_tensor_count": expected_lora_tensor_count,
            "expected_parameter_count": expected_lora_parameters,
            "exact_injection_checks": exact_lora_injection_checks,
        },
        "gradient_checkpointing_supported": checkpointing_supported,
        "gradient_checkpointing_enabled": checkpointing_enabled,
        "readout": "CLS + mean(true patches) -> LN -> Linear512 -> GELU -> Dropout0.30",
        "auxiliary_head_inference": False,
        "parameter_counts": {
            "total": total_parameters,
            "trainable": trainable_parameters,
            "lora": lora_parameters,
            "trainable_fraction": trainable_parameters / total_parameters,
        },
    }
    target_audit["injection_validation"] = {
        "passed": True,
        "lora_trainable_parameter_names": lora_trainables,
        "final_norm_trainable_parameter_names": final_norm_trainables,
        "unintended_backbone_trainables": unintended_backbone_trainables,
        "parameter_counts": runtime_recipe["parameter_counts"],
        "exact_lora_injection_checks": exact_lora_injection_checks,
        "expected_lora_trainable_tensor_count": expected_lora_tensor_count,
        "expected_lora_parameter_count": expected_lora_parameters,
    }
    if accelerator.is_main_process:
        write_json_atomic(target_audit, ARTIFACT_DIR / "lora_target_audit.json")
        write_json_atomic(
            signed_document({
                "model_id": MODEL_ID,
                "model_revision": RESOLVED_MODEL_REVISION,
                "architecture_class": type(underlying).__name__,
                "runtime_recipe": runtime_recipe,
            }),
            ARTIFACT_DIR / "model_manifest.json",
        )
    accelerator.wait_for_everyone()
    return model, processor, runtime_recipe


In [8]:
def load_rgb_image(path):
    try:
        with Image.open(path) as image:
            width, height = image.size
            if width <= 0 or height <= 0:
                raise ValueError("invalid image dimensions")
            image.draft("RGB", (1024, 1024))
            rgb = ImageOps.exif_transpose(image).convert("RGB")
            if width * height >= 8_000_000 or max(width, height) > 4096:
                rgb.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
            return rgb.copy()
    except Exception as exc:
        raise RuntimeError("Cannot decode official image {}".format(path)) from exc

def torchvision_interpolation(processor):
    resampling = Image.Resampling(int(processor.resample))
    mapping = {
        Image.Resampling.NEAREST: InterpolationMode.NEAREST,
        Image.Resampling.BILINEAR: InterpolationMode.BILINEAR,
        Image.Resampling.BICUBIC: InterpolationMode.BICUBIC,
        Image.Resampling.LANCZOS: InterpolationMode.LANCZOS,
        Image.Resampling.BOX: InterpolationMode.BOX,
        Image.Resampling.HAMMING: InterpolationMode.HAMMING,
    }
    if resampling not in mapping:
        raise RuntimeError("Official processor interpolation is unsupported.")
    return mapping[resampling]

def make_transforms(processor):
    interpolation = torchvision_interpolation(processor)
    mean = tuple(float(value) for value in processor.image_mean)
    std = tuple(float(value) for value in processor.image_std)
    train_transform = v2.Compose([
        v2.ToImage(),
        v2.RandomResizedCrop(
            (IMAGE_SIZE, IMAGE_SIZE), scale=(0.65, 1.0), ratio=(0.8, 1.25),
            interpolation=interpolation, antialias=True,
        ),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomRotation(degrees=12, interpolation=interpolation),
        v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
        v2.RandomGrayscale(p=0.10),
        v2.RandomApply([v2.GaussianBlur(kernel_size=5, sigma=(0.1, 1.2))], p=0.10),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=mean, std=std),
        v2.RandomErasing(p=0.15, scale=(0.02, 0.12), ratio=(0.5, 2.0), value=0),
    ])
    deterministic_transform = v2.Compose([
        v2.ToImage(),
        v2.Resize(
            (IMAGE_SIZE, IMAGE_SIZE), interpolation=interpolation, antialias=True
        ),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=mean, std=std),
    ])
    return train_transform, deterministic_transform

class ImageRows(Dataset):
    def __init__(self, frame, indices, transform):
        self.frame = frame
        self.indices = [int(index) for index in indices]
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        row_index = self.indices[position]
        row = self.frame.iloc[row_index]
        image = self.transform(load_rgb_image(row["path"]))
        if tuple(image.shape) != (3, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError("Transformed image shape failed for {}".format(row["image_id"]))
        return {
            "pixel_values": image,
            "label": int(row["label"]),
            "row_index": row_index,
            "image_id": str(row["image_id"]),
        }

class ExactDistributedEvalSampler(Sampler):
    def __init__(self, dataset_size, rank, world_size):
        self.indices = list(range(int(dataset_size)))[int(rank)::int(world_size)]

    def __iter__(self):
        return iter(self.indices)

    def __len__(self):
        return len(self.indices)

def seed_loader_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def make_fold_loaders(processor, fold, accelerator):
    train_transform, deterministic_transform = make_transforms(processor)
    training_indices = np.flatnonzero(train_df["fold"].to_numpy() != fold)
    validation_indices = np.flatnonzero(train_df["fold"].to_numpy() == fold)
    train_dataset = ImageRows(train_df, training_indices, train_transform)
    validation_dataset = ImageRows(train_df, validation_indices, deterministic_transform)
    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=accelerator.num_processes,
        rank=accelerator.process_index,
        shuffle=True,
        seed=SEED + 100 * fold,
        drop_last=False,
    )
    validation_sampler = ExactDistributedEvalSampler(
        len(validation_dataset), accelerator.process_index, accelerator.num_processes
    )
    generator = torch.Generator().manual_seed(SEED + 1000 * fold + accelerator.process_index)
    loader_kwargs = {
        "num_workers": LOADER_WORKERS_PER_PROCESS,
        "pin_memory": True,
        "persistent_workers": False,
        "worker_init_fn": seed_loader_worker,
    }
    train_loader = DataLoader(
        train_dataset, batch_size=MICROBATCH_PER_GPU, sampler=train_sampler,
        drop_last=False, generator=generator, **loader_kwargs,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=MICROBATCH_PER_GPU, sampler=validation_sampler,
        shuffle=False, drop_last=False, **loader_kwargs,
    )
    return (
        train_loader, validation_loader, train_sampler,
        training_indices, validation_indices, deterministic_transform,
    )

def class_weight_values(training_indices):
    counts = np.bincount(
        train_df.iloc[training_indices]["label"].to_numpy(dtype=np.int64),
        minlength=NUM_CLASSES,
    )
    total = int(counts.sum())
    weights = total / (NUM_CLASSES * counts.astype(np.float64))
    return weights.astype(np.float32), counts

def balanced_binary_focal_loss(logits, labels, ro_counts):
    mask = labels != 1
    if not mask.any():
        return logits.sum() * 0.0
    targets = (labels[mask] == 2).float()
    selected_logits = logits[mask]
    recyclable_count = float(ro_counts[0])
    organic_count = float(ro_counts[2])
    ro_total = recyclable_count + organic_count
    negative_weight = ro_total / (2.0 * recyclable_count)
    positive_weight = ro_total / (2.0 * organic_count)
    bce = F.binary_cross_entropy_with_logits(selected_logits, targets, reduction="none")
    probability_true = torch.where(
        targets > 0.5, selected_logits.sigmoid(), (-selected_logits).sigmoid()
    )
    balance = torch.where(
        targets > 0.5,
        torch.full_like(targets, positive_weight),
        torch.full_like(targets, negative_weight),
    )
    return (balance * (1.0 - probability_true).pow(FOCAL_GAMMA) * bce).mean()

def loss_components(output, labels, class_weights, ro_counts):
    main = F.cross_entropy(
        output["main_logits"], labels, weight=class_weights,
        label_smoothing=LABEL_SMOOTHING,
    )
    auxiliary = balanced_binary_focal_loss(
        output["auxiliary_logits"], labels, ro_counts
    )
    return main + AUXILIARY_WEIGHT * auxiliary, main, auxiliary

def build_optimizer(model):
    groups = {
        "lora_decay": {"params": [], "lr": LORA_LR, "weight_decay": WEIGHT_DECAY},
        "lora_no_decay": {"params": [], "lr": LORA_LR, "weight_decay": 0.0},
        "norm_head_decay": {"params": [], "lr": NORM_HEAD_LR, "weight_decay": WEIGHT_DECAY},
        "norm_head_no_decay": {"params": [], "lr": NORM_HEAD_LR, "weight_decay": 0.0},
    }
    names = {key: [] for key in groups}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if id(parameter) in seen:
            raise RuntimeError("Trainable parameter appears twice.")
        seen.add(id(parameter))
        family = "lora" if "lora_" in name else "norm_head"
        no_decay = parameter.ndim == 1 or name.endswith(".bias") or "norm" in name.lower()
        key = family + ("_no_decay" if no_decay else "_decay")
        groups[key]["params"].append(parameter)
        names[key].append(name)
    expected = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
    if (
        seen != expected
        or not groups["lora_decay"]["params"]
        or not groups["norm_head_decay"]["params"]
        or not groups["norm_head_no_decay"]["params"]
    ):
        raise RuntimeError("Optimizer groups do not exactly cover trainable parameters.")
    active_groups = [group for group in groups.values() if group["params"]]
    optimizer = torch.optim.AdamW(active_groups, betas=(0.9, 0.999), eps=1e-8)
    manifest = {
        key: {
            "learning_rate": groups[key]["lr"],
            "weight_decay": groups[key]["weight_decay"],
            "parameter_count": sum(parameter.numel() for parameter in groups[key]["params"]),
            "parameter_names": names[key],
        }
        for key in groups
    }
    return optimizer, manifest


## 12. Fold-0 runtime preflight and one forward/backward memory test

Both workers verify two T4 processes, target agreement, processor values, dynamic patch/register layout, positional interpolation at 320px, official final-norm equivalence through a hook, finite tensors, nonzero intended gradients, and measured peak allocation. The probe performs no optimizer step and restores RNG state.

In [9]:
def final_norm_from_classifier(classifier):
    return resolve_module(
        classifier.backbone.get_base_model(), classifier.final_norm_path
    )

def run_fold0_preflight(model, processor, runtime_recipe, training_indices, accelerator):
    unwrapped = accelerator.unwrap_model(model)
    class_zero = [
        int(index) for index in training_indices
        if int(train_df.iloc[index]["label"]) == 0
    ]
    class_two = [
        int(index) for index in training_indices
        if int(train_df.iloc[index]["label"]) == 2
    ]
    rank = accelerator.process_index
    probe_indices = [class_zero[rank], class_two[rank]]
    _, deterministic_transform = make_transforms(processor)
    probe_loader = DataLoader(
        ImageRows(train_df, probe_indices, deterministic_transform),
        batch_size=MICROBATCH_PER_GPU, shuffle=False, num_workers=0,
    )
    batch = next(iter(probe_loader))

    python_state = random.getstate()
    numpy_state = np.random.get_state()
    cpu_rng_state = torch.get_rng_state()
    device_rng_state = torch.cuda.get_rng_state(accelerator.device)
    torch.cuda.reset_peak_memory_stats(accelerator.device)
    capture = {"count": 0, "output": None}

    def final_norm_hook(module, inputs, output):
        capture["count"] += 1
        capture["output"] = output.detach()

    handle = final_norm_from_classifier(unwrapped).register_forward_hook(final_norm_hook)
    model.train()
    pixels = batch["pixel_values"].to(accelerator.device, non_blocking=True)
    labels = batch["label"].to(accelerator.device, non_blocking=True)
    training_weights, training_counts = class_weight_values(training_indices)
    weight_tensor = torch.tensor(training_weights, device=accelerator.device)
    try:
        with accelerator.autocast():
            output = model(pixels, return_tokens=True)
            total_loss, main_loss, auxiliary_loss = loss_components(
                output, labels, weight_tensor, training_counts
            )
        accelerator.backward(total_loss)
        tokens = output["tokens"]
        patches = output["true_patch_tokens"]
        hook_difference = float(
            (capture["output"].float() - tokens.detach().float()).abs().max().item()
        )
        gradient_groups = {"lora": [], "final_norm": [], "head": []}
        final_norm_ids = {
            id(parameter) for parameter in final_norm_from_classifier(unwrapped).parameters()
        }
        for name, parameter in unwrapped.named_parameters():
            if not parameter.requires_grad:
                continue
            if "lora_" in name:
                group = "lora"
            elif id(parameter) in final_norm_ids:
                group = "final_norm"
            else:
                group = "head"
            gradient_groups[group].append({
                "name": name,
                "finite": bool(parameter.grad is not None and torch.isfinite(parameter.grad).all()),
                "nonzero": bool(parameter.grad is not None and parameter.grad.detach().abs().sum() > 0),
            })
        peak_gib = torch.cuda.max_memory_allocated(accelerator.device) / (2 ** 30)
        properties = torch.cuda.get_device_properties(accelerator.device)
        total_gib = properties.total_memory / (2 ** 30)
        local_report = {
            "rank": rank,
            "device": str(accelerator.device),
            "gpu_name": properties.name,
            "total_vram_gib": total_gib,
            "peak_allocated_gib": peak_gib,
            "post_probe_headroom_gib": total_gib - peak_gib,
            "losses": {
                "total": float(total_loss.detach()),
                "main": float(main_loss.detach()),
                "auxiliary": float(auxiliary_loss.detach()),
            },
            "token_shape": list(tokens.shape),
            "patch_shape": list(patches.shape),
            "final_norm_hook_calls": capture["count"],
            "final_norm_output_max_difference": hook_difference,
            "finite_tokens": bool(torch.isfinite(tokens).all()),
            "finite_patches": bool(torch.isfinite(patches).all()),
            "gradient_groups": gradient_groups,
        }
        local_report["checks"] = {
            "t4": "T4" in properties.name.upper(),
            "token_shape_dynamic": (
                tokens.shape[1]
                == 1 + runtime_recipe["num_register_tokens"] + runtime_recipe["true_patch_count"]
            ),
            "true_patch_count": patches.shape[1] == runtime_recipe["true_patch_count"],
            "registers_excluded": (
                tokens.shape[1] - patches.shape[1]
                == 1 + runtime_recipe["num_register_tokens"]
            ),
            "official_final_norm_once": capture["count"] == 1 and hook_difference <= 1e-6,
            "positional_interpolation_320_succeeded": tokens.shape[1] > patches.shape[1],
            "finite": bool(torch.isfinite(tokens).all() and torch.isfinite(total_loss)),
            "intended_gradients": all(
                items
                and all(item["finite"] for item in items)
                and any(item["nonzero"] for item in items)
                for items in gradient_groups.values()
            ),
            "memory_headroom_at_least_0_75_gib": total_gib - peak_gib >= 0.75,
        }
    finally:
        handle.remove()
        model.zero_grad(set_to_none=True)
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        torch.set_rng_state(cpu_rng_state)
        torch.cuda.set_rng_state(device_rng_state, accelerator.device)

    gathered = [None] * accelerator.num_processes
    dist.all_gather_object(gathered, local_report)
    passed = all(all(report["checks"].values()) for report in gathered)
    report = {
        "passed": passed,
        "exactly_two_processes": accelerator.num_processes == 2,
        "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
        "workers": gathered,
        "model_contract": {
            "official_final_normalized_output": "last_hidden_state",
            "cls_index": 0,
            "register_count": runtime_recipe["num_register_tokens"],
            "true_patch_start": 1 + runtime_recipe["num_register_tokens"],
            "true_patch_count": runtime_recipe["true_patch_count"],
            "patch_grid": runtime_recipe["patch_grid"],
        },
    }
    report["preflight_signature"] = sha256_json(report)
    if accelerator.is_main_process:
        write_json_atomic(report, ARTIFACT_DIR / "preflight_report.json")
    accelerator.wait_for_everyone()
    if not passed or accelerator.num_processes != 2:
        raise RuntimeError("Fold-0 forward/backward preflight failed: {}".format(report))
    return report


## 13. Fold-0 training and persisted development decision

The fold trainer is shared by all folds, but orchestration calls fold 0 first. Validation uses an exact non-padding distributed sampler and one object collective after local inference, so probabilities, labels, row indices, and image IDs are gathered without padding duplicates. Completed folds are reused only after signature validation and fresh checkpoint reproduction.

In [10]:
def distributed_predict(model, loader, expected_indices, accelerator):
    model.eval()
    local_indices, local_ids, local_labels, local_probabilities = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            pixels = batch["pixel_values"].to(accelerator.device, non_blocking=True)
            with accelerator.autocast():
                logits = model(pixels)["main_logits"]
            probabilities = logits.float().softmax(dim=1).cpu().numpy()
            local_probabilities.append(probabilities)
            local_labels.extend(batch["label"].numpy().astype(np.int64).tolist())
            local_indices.extend(batch["row_index"].numpy().astype(np.int64).tolist())
            local_ids.extend([str(value) for value in batch["image_id"]])
    payload = {
        "indices": local_indices,
        "ids": local_ids,
        "labels": local_labels,
        "probabilities": (
            np.concatenate(local_probabilities, axis=0)
            if local_probabilities else np.empty((0, NUM_CLASSES), dtype=np.float32)
        ),
    }
    gathered = [None] * accelerator.num_processes
    dist.all_gather_object(gathered, payload)
    if not accelerator.is_main_process:
        return None, None, None

    indices = np.asarray(
        [value for part in gathered for value in part["indices"]], dtype=np.int64
    )
    labels = np.asarray(
        [value for part in gathered for value in part["labels"]], dtype=np.int64
    )
    ids = [value for part in gathered for value in part["ids"]]
    probabilities = np.concatenate(
        [part["probabilities"] for part in gathered], axis=0
    )
    order = np.argsort(indices, kind="stable")
    indices, labels, probabilities = indices[order], labels[order], probabilities[order]
    ids = [ids[position] for position in order]
    expected = np.asarray(expected_indices, dtype=np.int64)
    exact = {
        "row_count": len(indices) == len(expected),
        "no_duplicate_indices": len(np.unique(indices)) == len(indices),
        "canonical_indices": np.array_equal(indices, expected),
        "labels_exact": np.array_equal(labels, train_df.iloc[expected]["label"].to_numpy()),
        "ids_exact": ids == train_df.iloc[expected]["image_id"].astype(str).tolist(),
    }
    if not all(exact.values()):
        raise RuntimeError("Distributed validation gather failed: {}".format(exact))
    probabilities = normalize_probabilities(probabilities)
    metrics = metric_bundle(labels, probabilities)
    metrics["gather_checks"] = exact
    metrics["row_order_verified"] = True
    return metrics, probabilities, ids

def fold_paths(fold):
    prefix = FOLDS_DIR / "fold_{}".format(fold)
    return {
        "adapter": Path(str(prefix) + "_lora_adapter.pt"),
        "head": Path(str(prefix) + "_heads_and_norm.pt"),
        "history": Path(str(prefix) + "_training_history.csv"),
        "metrics": Path(str(prefix) + "_metrics.json"),
        "report": Path(str(prefix) + "_classification_report.csv"),
        "predictions": Path(str(prefix) + "_validation_predictions.csv"),
        "probabilities": Path(str(prefix) + "_validation_probabilities.npy"),
        "complete": Path(str(prefix) + "_complete.json"),
    }

def checkpoint_metadata(fold, fold_signature, runtime_recipe, epoch, metrics):
    return {
        "experiment": EXPERIMENT_NAME,
        "fold": int(fold),
        "model_id": MODEL_ID,
        "model_revision": RESOLVED_MODEL_REVISION,
        "fold_assignment_signature": FOLD_ASSIGNMENT_SIGNATURE,
        "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
        "fold_artifact_signature": fold_signature,
        "best_epoch": int(epoch),
        "validation_rows": int(metrics["rows"]),
        "selection_metrics": {
            key: metrics[key]
            for key in ("macro_f1", "total_errors", "ro_errors", "f1_electronic", "log_loss")
        },
    }

def save_best_fold(model, fold, fold_signature, runtime_recipe, epoch, metrics, probabilities, validation_indices):
    paths = fold_paths(fold)
    classifier = model
    metadata = checkpoint_metadata(
        fold, fold_signature, runtime_recipe, epoch, metrics
    )
    adapter_state = {
        key: value.detach().cpu()
        for key, value in get_peft_model_state_dict(
            classifier.backbone, save_embedding_layers=False
        ).items()
    }
    atomic_torch_save(
        {"metadata": metadata, "state_dict": adapter_state},
        paths["adapter"],
    )
    atomic_torch_save(
        {
            "metadata": metadata,
            "features": cpu_state_dict(classifier.features),
            "main_classifier": cpu_state_dict(classifier.main_classifier),
            "auxiliary_classifier": cpu_state_dict(classifier.auxiliary_classifier),
            "final_norm": cpu_state_dict(final_norm_from_classifier(classifier)),
        },
        paths["head"],
    )
    probabilities = normalize_probabilities(probabilities)
    write_npy_atomic(probabilities, paths["probabilities"])
    labels = train_df.iloc[validation_indices]["label"].to_numpy(dtype=np.int64)
    predictions = probabilities.argmax(axis=1)
    prediction_frame = pd.DataFrame({
        "fold_artifact_signature": fold_signature,
        "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
        "row_index": validation_indices,
        "image_id": train_df.iloc[validation_indices]["image_id"].astype(str).tolist(),
        "label": labels,
        "prediction": predictions,
        "p_recyclable": probabilities[:, 0],
        "p_electronic": probabilities[:, 1],
        "p_organic": probabilities[:, 2],
    })
    write_csv_atomic(prediction_frame, paths["predictions"])
    write_json_atomic(
        signed_document({**metrics, "best_epoch": int(epoch), "fold": int(fold),
                         "fold_artifact_signature": fold_signature}),
        paths["metrics"],
    )
    report = classification_report(
        labels, predictions, labels=[0, 1, 2], target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    write_csv_atomic(pd.DataFrame(report).transpose(), paths["report"], index=True)

def load_fold_weights(classifier, fold, expected_signature):
    paths = fold_paths(fold)
    adapter = torch.load(paths["adapter"], map_location="cpu", weights_only=False)
    head = torch.load(paths["head"], map_location="cpu", weights_only=False)
    for payload in (adapter, head):
        metadata = payload.get("metadata", {})
        checks = {
            "fold": metadata.get("fold") == fold,
            "revision": metadata.get("model_revision") == RESOLVED_MODEL_REVISION,
            "recipe": metadata.get("runtime_recipe_signature") == expected_signature,
            "fold_signature": bool(metadata.get("fold_artifact_signature")),
        }
        if not all(checks.values()):
            raise RuntimeError("Checkpoint metadata mismatch: {}".format(checks))
    if adapter["metadata"] != head["metadata"]:
        raise RuntimeError("Adapter and head checkpoint metadata differ.")
    set_peft_model_state_dict(classifier.backbone, adapter["state_dict"])
    classifier.features.load_state_dict(head["features"], strict=True)
    classifier.main_classifier.load_state_dict(head["main_classifier"], strict=True)
    classifier.auxiliary_classifier.load_state_dict(head["auxiliary_classifier"], strict=True)
    final_norm_from_classifier(classifier).load_state_dict(head["final_norm"], strict=True)
    return adapter["metadata"]

def update_reproduction_record(fold, max_difference, passed):
    path = ARTIFACT_DIR / "checkpoint_reproduction_check.json"
    document = read_json(path) if path.is_file() else {"fold_validation_reload": {}}
    document["fold_validation_reload"][str(fold)] = {
        "max_absolute_probability_difference": float(max_difference),
        "tolerance": RELOAD_TOLERANCE,
        "passed": bool(passed),
    }
    document["all_available_fold_reloads_passed"] = all(
        item["passed"] for item in document["fold_validation_reload"].values()
    )
    document["signature"] = sha256_json({
        key: value for key, value in document.items() if key != "signature"
    })
    write_json_atomic(document, path)

def completed_fold_is_reusable(fold, fold_signature, runtime_recipe, validation_indices):
    paths = fold_paths(fold)
    if not paths["complete"].is_file():
        return False
    complete = read_json(paths["complete"])
    required = [
        paths["adapter"], paths["head"], paths["history"], paths["metrics"],
        paths["report"], paths["predictions"], paths["probabilities"],
    ]
    files_exist = all(path.is_file() for path in required)
    checks = {
        "signed": verify_signed_document(complete),
        "fold": complete.get("fold") == fold,
        "fold_signature": complete.get("fold_artifact_signature") == fold_signature,
        "runtime_recipe": (
            complete.get("runtime_recipe_signature")
            == runtime_recipe["runtime_recipe_signature"]
        ),
        "revision": complete.get("model_revision") == RESOLVED_MODEL_REVISION,
        "rows": complete.get("validation_rows") == len(validation_indices),
        "files_exist": files_exist,
        "adapter_sha256": files_exist and complete.get("adapter_sha256") == sha256_file(paths["adapter"]),
        "head_sha256": files_exist and complete.get("head_sha256") == sha256_file(paths["head"]),
        "probabilities_sha256": (
            files_exist
            and complete.get("probabilities_sha256") == sha256_file(paths["probabilities"])
        ),
        "history_sha256": files_exist and complete.get("history_sha256") == sha256_file(paths["history"]),
        "metrics_sha256": files_exist and complete.get("metrics_sha256") == sha256_file(paths["metrics"]),
        "report_sha256": files_exist and complete.get("report_sha256") == sha256_file(paths["report"]),
        "predictions_sha256": (
            files_exist
            and complete.get("predictions_sha256") == sha256_file(paths["predictions"])
        ),
    }
    if not all(checks.values()):
        raise RuntimeError("Existing completed fold is incompatible: {}".format(checks))
    return True

def run_fold(fold, accelerator, preflight=False):
    fold_seed = SEED + 100 * fold
    set_seed(fold_seed, device_specific=False)
    model, processor, runtime_recipe = build_runtime_model(accelerator)
    (
        train_loader, validation_loader, train_sampler,
        training_indices, validation_indices, _,
    ) = make_fold_loaders(processor, fold, accelerator)
    class_weights, class_counts = class_weight_values(training_indices)
    optimizer, optimizer_manifest = build_optimizer(model)
    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = max(1, int(round(WARMUP_RATIO * total_updates)))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_updates, num_training_steps=total_updates
    )

    runtime_recipe = {
        **runtime_recipe,
        "augmentation": BASE_CONFIG["augmentation"],
        "loss": {
            "main": "class-weighted CE", "class_weight_formula": "total/(3*class_count)",
            "label_smoothing": LABEL_SMOOTHING,
            "auxiliary": "balanced binary focal on class 0/2 only",
            "focal_gamma": FOCAL_GAMMA, "auxiliary_weight": AUXILIARY_WEIGHT,
            "auxiliary_inference": False,
        },
        "optimizer_groups": optimizer_manifest,
        "scheduler": {"name": "cosine", "warmup_ratio": WARMUP_RATIO},
        "batching": {
            "processes": 2, "microbatch_per_gpu": MICROBATCH_PER_GPU,
            "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
            "effective_batch": EFFECTIVE_BATCH,
        },
        "seeds": {"global": SEED, "fold_formula": "42 + 100 * fold"},
        "epochs": {"maximum": MAX_EPOCHS, "early_stopping_patience": EARLY_STOPPING_PATIENCE},
        "selection_order": [
            "higher Macro F1", "fewer errors", "fewer R/O errors", "lower normalized log loss"
        ],
        "development_thresholds": DEVELOPMENT_THRESHOLDS,
        "adoption_thresholds": ADOPTION_THRESHOLDS,
        "fold_assignment_signature": FOLD_ASSIGNMENT_SIGNATURE,
    }
    runtime_recipe["runtime_recipe_signature"] = sha256_json(runtime_recipe)
    signatures = [None] * accelerator.num_processes
    dist.all_gather_object(signatures, runtime_recipe["runtime_recipe_signature"])
    if len(set(signatures)) != 1:
        raise RuntimeError("Workers built different runtime recipes.")

    # The scheduler is intentionally not wrapped: its horizon is already based on
    # this rank's explicit DistributedSampler loader, so one scheduler step equals one update.
    model, optimizer = accelerator.prepare(model, optimizer)
    class_weight_tensor = torch.tensor(class_weights, device=accelerator.device)
    if preflight:
        run_fold0_preflight(
            model, processor, runtime_recipe, training_indices, accelerator
        )

    fold_signature = sha256_json({
        "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
        "fold": fold,
        "validation_ids_sha256": sha256_json(
            train_df.iloc[validation_indices]["image_id"].astype(str).tolist()
        ),
        "training_rows": len(training_indices),
        "validation_rows": len(validation_indices),
    })
    reusable = completed_fold_is_reusable(
        fold, fold_signature, runtime_recipe, validation_indices
    )
    reusable = broadcast_object(accelerator, reusable if accelerator.is_main_process else None)
    unwrapped = accelerator.unwrap_model(model)

    if reusable:
        load_fold_weights(
            unwrapped, fold, runtime_recipe["runtime_recipe_signature"]
        )
        accelerator.wait_for_everyone()
        metrics, reproduced, _ = distributed_predict(
            model, validation_loader, validation_indices, accelerator
        )
        if accelerator.is_main_process:
            persisted = np.load(
                fold_paths(fold)["probabilities"], allow_pickle=False
            )
            difference = float(np.max(np.abs(persisted - reproduced)))
            passed = difference <= RELOAD_TOLERANCE
            update_reproduction_record(fold, difference, passed)
            if not passed:
                raise RuntimeError("Reusable fold failed fresh probability reproduction.")
            stored = read_json(fold_paths(fold)["metrics"])
            stored.pop("signature", None)
            metrics["best_epoch"] = int(stored["best_epoch"])
            print("Fold {} resumed after signed reload verification.".format(fold))
        metrics = broadcast_object(accelerator, metrics)
        accelerator.wait_for_everyone()
        del model, optimizer, scheduler, train_loader, validation_loader
        accelerator.free_memory()
        gc.collect()
        torch.cuda.empty_cache()
        return metrics, runtime_recipe

    history = []
    best_metrics = None
    best_epoch = None
    epochs_without_improvement = 0
    fold_started = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_started = time.perf_counter()
        model.train()
        train_sampler.set_epoch(epoch)
        running = torch.zeros(4, dtype=torch.float64, device=accelerator.device)
        number_of_batches = len(train_loader)
        for step, batch in enumerate(train_loader):
            pixels = batch["pixel_values"].to(accelerator.device, non_blocking=True)
            labels = batch["label"].to(accelerator.device, non_blocking=True)
            position_in_group = step % GRAD_ACCUM_STEPS
            group_start = step - position_in_group
            group_size = min(GRAD_ACCUM_STEPS, number_of_batches - group_start)
            should_step = position_in_group + 1 == group_size
            synchronization = (
                contextlib.nullcontext()
                if should_step else accelerator.no_sync(model)
            )
            with synchronization:
                with accelerator.autocast():
                    output = model(pixels)
                    total_loss, main_loss, auxiliary_loss = loss_components(
                        output, labels, class_weight_tensor, class_counts
                    )
                    # Accelerator.backward divides by its configured accumulation count.
                    backward_loss = total_loss * (GRAD_ACCUM_STEPS / group_size)
                accelerator.backward(backward_loss)
            batch_size = labels.shape[0]
            running[0] += total_loss.detach().double() * batch_size
            running[1] += main_loss.detach().double() * batch_size
            running[2] += auxiliary_loss.detach().double() * batch_size
            running[3] += batch_size
            if should_step:
                accelerator.clip_grad_norm_(
                    [parameter for parameter in model.parameters() if parameter.requires_grad],
                    GRAD_CLIP,
                )
                optimizer.step()
                if not optimizer.step_was_skipped:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            if (step + 1) % 100 == 0:
                if not torch.isfinite(total_loss.detach()):
                    raise RuntimeError("Nonfinite training loss at fold {} epoch {} batch {}.".format(
                        fold, epoch, step + 1
                    ))
            if (
                accelerator.is_main_process
                and (step + 1) % 100 == 0
            ):
                print(
                    "fold={} epoch={} batch={}/{} elapsed_min={:.1f}".format(
                        fold, epoch, step + 1, number_of_batches,
                        (time.perf_counter() - epoch_started) / 60.0,
                    ),
                    flush=True,
                )
        dist.all_reduce(running, op=dist.ReduceOp.SUM)
        train_total, train_main, train_auxiliary = (
            float(running[index] / running[3]) for index in range(3)
        )
        metrics, probabilities, _ = distributed_predict(
            model, validation_loader, validation_indices, accelerator
        )
        metrics = broadcast_object(accelerator, metrics)
        epoch_seconds = time.perf_counter() - epoch_started

        if accelerator.is_main_process:
            improved = best_metrics is None or metric_rank(metrics) > metric_rank(best_metrics)
            if improved:
                best_metrics = dict(metrics)
                best_epoch = epoch
                epochs_without_improvement = 0
                save_best_fold(
                    unwrapped, fold, fold_signature, runtime_recipe,
                    epoch, metrics, probabilities, validation_indices,
                )
            else:
                epochs_without_improvement += 1
            row = {
                "fold": fold, "epoch": epoch,
                "fold_artifact_signature": fold_signature,
                "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
                "train_total_loss": train_total,
                "train_main_loss": train_main,
                "train_auxiliary_loss": train_auxiliary,
                **{key: metrics[key] for key in (
                    "macro_f1", "total_errors", "ro_errors",
                    "f1_electronic", "log_loss"
                )},
                "epoch_seconds": epoch_seconds,
                "elapsed_fold_seconds": time.perf_counter() - fold_started,
                "estimated_remaining_epoch_seconds": epoch_seconds * (MAX_EPOCHS - epoch),
                "improved": improved,
            }
            history.append(row)
            print(
                "fold={fold} epoch={epoch}/{maximum} loss={loss:.5f} "
                "main={main:.5f} aux={aux:.5f} val_f1={f1:.9f} "
                "errors={errors} ro={ro} electronic_f1={electronic:.9f} "
                "elapsed_min={elapsed:.1f} eta_min={eta:.1f}".format(
                    fold=fold, epoch=epoch, maximum=MAX_EPOCHS,
                    loss=train_total, main=train_main, aux=train_auxiliary,
                    f1=metrics["macro_f1"], errors=metrics["total_errors"],
                    ro=metrics["ro_errors"], electronic=metrics["f1_electronic"],
                    elapsed=epoch_seconds / 60.0,
                    eta=epoch_seconds * (MAX_EPOCHS - epoch) / 60.0,
                ),
                flush=True,
            )
            state = {
                "improved": improved,
                "best_epoch": best_epoch,
                "epochs_without_improvement": epochs_without_improvement,
                "stop": epochs_without_improvement >= EARLY_STOPPING_PATIENCE,
            }
        else:
            state = None
        state = broadcast_object(accelerator, state)
        accelerator.wait_for_everyone()
        if state["stop"]:
            break

    if accelerator.is_main_process:
        write_csv_atomic(pd.DataFrame(history), fold_paths(fold)["history"])
    accelerator.wait_for_everyone()
    load_fold_weights(unwrapped, fold, runtime_recipe["runtime_recipe_signature"])
    accelerator.wait_for_everyone()
    reproduced_metrics, reproduced_probabilities, _ = distributed_predict(
        model, validation_loader, validation_indices, accelerator
    )
    if accelerator.is_main_process:
        persisted_probabilities = np.load(
            fold_paths(fold)["probabilities"], allow_pickle=False
        )
        difference = float(np.max(np.abs(
            persisted_probabilities - reproduced_probabilities
        )))
        identical_predictions = np.array_equal(
            persisted_probabilities.argmax(axis=1),
            reproduced_probabilities.argmax(axis=1),
        )
        passed = difference <= RELOAD_TOLERANCE and identical_predictions
        update_reproduction_record(fold, difference, passed)
        if not passed:
            raise RuntimeError("Fold {} checkpoint reload failed.".format(fold))
        complete = signed_document({
            "fold": fold,
            "fold_artifact_signature": fold_signature,
            "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
            "model_revision": RESOLVED_MODEL_REVISION,
            "validation_rows": len(validation_indices),
            "best_epoch": best_epoch,
            "adapter_sha256": sha256_file(fold_paths(fold)["adapter"]),
            "head_sha256": sha256_file(fold_paths(fold)["head"]),
            "probabilities_sha256": sha256_file(fold_paths(fold)["probabilities"]),
            "history_sha256": sha256_file(fold_paths(fold)["history"]),
            "metrics_sha256": sha256_file(fold_paths(fold)["metrics"]),
            "report_sha256": sha256_file(fold_paths(fold)["report"]),
            "predictions_sha256": sha256_file(fold_paths(fold)["predictions"]),
            "reload_max_probability_difference": difference,
            "reload_tolerance": RELOAD_TOLERANCE,
        })
        write_json_atomic(complete, fold_paths(fold)["complete"])
        best_metrics["best_epoch"] = best_epoch
    best_metrics = broadcast_object(accelerator, best_metrics)
    accelerator.wait_for_everyone()
    del model, optimizer, scheduler, train_loader, validation_loader
    accelerator.free_memory()
    gc.collect()
    torch.cuda.empty_cache()
    return best_metrics, runtime_recipe


## 14. Conditional signed recipe lock

Only a development pass can create locked_recipe_before_assessment.json. It freezes the complete runtime-discovered recipe before any assessment fold.

In [11]:
def make_development_decision(metrics, runtime_recipe):
    reference = V10_FOLD_REFERENCE[0]
    reproduction = read_json(ARTIFACT_DIR / "checkpoint_reproduction_check.json")
    target_audit = read_json(ARTIFACT_DIR / "lora_target_audit.json")
    checks = {
        "macro_f1_at_least_0_9910": metrics["macro_f1"] >= DEVELOPMENT_THRESHOLDS["min_macro_f1"],
        "gain_over_v10_at_least_0_0010": (
            metrics["macro_f1"] - reference["macro_f1"]
            >= DEVELOPMENT_THRESHOLDS["min_gain_over_v10_fold0"]
        ),
        "total_errors_at_most_57": metrics["total_errors"] <= DEVELOPMENT_THRESHOLDS["max_total_errors"],
        "ro_errors_at_most_50": metrics["ro_errors"] <= DEVELOPMENT_THRESHOLDS["max_ro_errors"],
        "electronic_f1_decline_at_most_0_0005": (
            reference["f1_electronic"] - metrics["f1_electronic"]
            <= DEVELOPMENT_THRESHOLDS["max_electronic_f1_decline"]
        ),
        "finite_probabilities_and_predictions": metrics["probabilities_finite"],
        "row_order_fold_membership_and_ids_exact": all(metrics["gather_checks"].values()),
        "raw_train_only_no_test_or_external_supervision": (
            not TEST_ACCESS_STATE["test_images_accessed"]
            and not TEST_ACCESS_STATE["external_or_manual_labels_used"]
            and DATA_CONTRACT["external_images_or_labels"] is False
        ),
        "one_architecture_and_one_recipe": True,
        "lora_target_audit_passed": (
            target_audit.get("injection_validation", {}).get("passed") is True
        ),
        "fold0_checkpoint_reload_passed": (
            reproduction["fold_validation_reload"]["0"]["passed"] is True
        ),
    }
    payload = {
        "status": "development_passed" if all(checks.values()) else "candidate_rejected",
        "fold": 0,
        "candidate_metrics": metrics,
        "v10_reference": reference,
        "macro_f1_gain": metrics["macro_f1"] - reference["macro_f1"],
        "electronic_f1_decline": reference["f1_electronic"] - metrics["f1_electronic"],
        "thresholds": DEVELOPMENT_THRESHOLDS,
        "checks": checks,
        "all_checks_passed": all(checks.values()),
        "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
        "test_access_state": dict(TEST_ACCESS_STATE),
    }
    return signed_document(payload)

def persist_recipe_lock(runtime_recipe, development_decision):
    if not development_decision["all_checks_passed"]:
        raise RuntimeError("Cannot lock an assessment recipe after development rejection.")
    payload = {
        "locked_before_assessment_folds": list(ASSESSMENT_FOLDS),
        "selection_source": "official raw train fold 0 only",
        "runtime_recipe": runtime_recipe,
        "runtime_recipe_signature": runtime_recipe["runtime_recipe_signature"],
        "development_decision_signature": development_decision["signature"],
        "test_access_state": dict(TEST_ACCESS_STATE),
        "recipe_changes_after_lock_allowed": False,
    }
    document = signed_document(payload)
    write_json_atomic(document, ARTIFACT_DIR / "locked_recipe_before_assessment.json")
    reloaded = read_json(ARTIFACT_DIR / "locked_recipe_before_assessment.json")
    if not verify_signed_document(reloaded):
        raise RuntimeError("Persisted assessment recipe lock failed signature validation.")
    return reloaded


## 15. Conditional folds 1–4

The distributed orchestrator below reaches this stage only after revalidating the signed fold-0 decision and recipe lock. Every fold reconstructs the identical adapter targets and training policy. No recipe mutation or control fallback exists.

## 16. Conditional OOF construction and train-only adoption decision

Fold probabilities are placed back into the canonical 26,527-row order. Adoption requires every absolute gate, at least three per-fold wins versus matching V10 references, bounded Electronic decline, bounded gain concentration, exact rows, and all reload checks.

In [12]:
def build_oof_and_selection(fold_metrics, locked_recipe):
    if not verify_signed_document(locked_recipe):
        raise RuntimeError("Recipe lock signature is invalid before OOF construction.")
    oof_probabilities = np.full((EXPECTED_TRAIN_ROWS, NUM_CLASSES), np.nan, dtype=np.float32)
    seen = np.zeros(EXPECTED_TRAIN_ROWS, dtype=np.int64)
    rows = []
    fold_gains = []
    for fold in range(N_FOLDS):
        paths = fold_paths(fold)
        complete = read_json(paths["complete"])
        if (
            not verify_signed_document(complete)
            or complete["runtime_recipe_signature"] != locked_recipe["runtime_recipe_signature"]
        ):
            raise RuntimeError("Fold {} completion signature failed.".format(fold))
        validation = pd.read_csv(paths["predictions"], dtype={"image_id": str})
        probabilities = np.load(paths["probabilities"], allow_pickle=False)
        expected_indices = np.flatnonzero(train_df["fold"].to_numpy() == fold)
        checks = {
            "rows": len(validation) == len(expected_indices) == len(probabilities),
            "indices": np.array_equal(validation["row_index"].to_numpy(), expected_indices),
            "ids": validation["image_id"].tolist()
            == train_df.iloc[expected_indices]["image_id"].astype(str).tolist(),
            "labels": np.array_equal(
                validation["label"].to_numpy(),
                train_df.iloc[expected_indices]["label"].to_numpy(),
            ),
            "finite": np.isfinite(probabilities).all(),
        }
        if not all(checks.values()):
            raise RuntimeError("Fold {} OOF placement failed: {}".format(fold, checks))
        oof_probabilities[expected_indices] = normalize_probabilities(probabilities)
        seen[expected_indices] += 1
        reference = V10_FOLD_REFERENCE[fold]
        gain = fold_metrics[fold]["macro_f1"] - reference["macro_f1"]
        fold_gains.append(gain)
        rows.append({
            "fold": fold,
            "rows": len(expected_indices),
            "best_epoch": fold_metrics[fold]["best_epoch"],
            "macro_f1": fold_metrics[fold]["macro_f1"],
            "v10_macro_f1": reference["macro_f1"],
            "macro_f1_gain": gain,
            "beats_matching_v10": gain > 0,
            "total_errors": fold_metrics[fold]["total_errors"],
            "ro_errors": fold_metrics[fold]["ro_errors"],
            "f1_electronic": fold_metrics[fold]["f1_electronic"],
            "reload_max_probability_difference": complete["reload_max_probability_difference"],
            "reload_passed": complete["reload_max_probability_difference"] <= RELOAD_TOLERANCE,
        })
    if not np.all(seen == 1) or not np.isfinite(oof_probabilities).all():
        raise RuntimeError("OOF rows are missing, duplicated, or nonfinite.")
    oof_probabilities = normalize_probabilities(oof_probabilities)
    labels = train_df["label"].to_numpy(dtype=np.int64)
    metrics = metric_bundle(labels, oof_probabilities)
    predictions = oof_probabilities.argmax(axis=1)
    oof_frame = pd.DataFrame({
        "row_index": np.arange(EXPECTED_TRAIN_ROWS),
        "image_id": train_df["image_id"].astype(str),
        "fold": train_df["fold"].astype(int),
        "label": labels,
        "prediction": predictions,
        "p_recyclable": oof_probabilities[:, 0],
        "p_electronic": oof_probabilities[:, 1],
        "p_organic": oof_probabilities[:, 2],
    })
    ro_errors = oof_frame[
        ((oof_frame["label"] == 0) & (oof_frame["prediction"] == 2))
        | ((oof_frame["label"] == 2) & (oof_frame["prediction"] == 0))
    ].copy()
    write_npy_atomic(oof_probabilities, ARTIFACT_DIR / "oof_probabilities.npy")
    write_csv_atomic(oof_frame, ARTIFACT_DIR / "oof_predictions.csv")
    write_csv_atomic(ro_errors, ARTIFACT_DIR / "oof_recyclable_organic_errors.csv")
    write_json_atomic(signed_document(metrics), ARTIFACT_DIR / "oof_metrics.json")
    write_csv_atomic(pd.DataFrame(rows), ARTIFACT_DIR / "assessment_per_fold.csv")

    positive_gains = [max(0.0, value) for value in fold_gains]
    positive_gain_sum = sum(positive_gains)
    largest_share = (
        max(positive_gains) / positive_gain_sum if positive_gain_sum > 0 else 1.0
    )
    reproduction = read_json(ARTIFACT_DIR / "checkpoint_reproduction_check.json")
    checks = {
        "oof_macro_f1_at_least_0_9910": metrics["macro_f1"] >= ADOPTION_THRESHOLDS["min_oof_macro_f1"],
        "oof_errors_at_most_280": metrics["total_errors"] <= ADOPTION_THRESHOLDS["max_oof_errors"],
        "oof_ro_errors_at_most_235": metrics["ro_errors"] <= ADOPTION_THRESHOLDS["max_oof_ro_errors"],
        "wins_at_least_3_of_5": sum(value > 0 for value in fold_gains) >= ADOPTION_THRESHOLDS["min_fold_wins"],
        "electronic_f1_decline_at_most_0_0005": (
            V10_OOF_REFERENCE["f1_electronic"] - metrics["f1_electronic"]
            <= ADOPTION_THRESHOLDS["max_electronic_f1_decline"]
        ),
        "single_fold_positive_gain_share_at_most_0_75": (
            largest_share <= ADOPTION_THRESHOLDS["max_single_positive_gain_share"]
        ),
        "all_fold_checkpoints_reload": (
            set(reproduction["fold_validation_reload"]) == {"0", "1", "2", "3", "4"}
            and all(item["passed"] for item in reproduction["fold_validation_reload"].values())
        ),
        "exact_complete_finite_oof": (
            len(oof_frame) == EXPECTED_TRAIN_ROWS
            and oof_frame["row_index"].is_unique
            and np.all(seen == 1)
            and np.isfinite(oof_probabilities).all()
        ),
        "no_test_access_before_signature": not any((
            TEST_ACCESS_STATE["test_images_accessed"],
            TEST_ACCESS_STATE["test_template_accessed"],
        )),
    }
    adopted = all(checks.values())
    aggregate = pd.DataFrame([{
        "candidate": "v12_vitl_qv_lora",
        "rows": EXPECTED_TRAIN_ROWS,
        "macro_f1": metrics["macro_f1"],
        "v10_macro_f1": V10_OOF_REFERENCE["macro_f1"],
        "macro_f1_gain": metrics["macro_f1"] - V10_OOF_REFERENCE["macro_f1"],
        "total_errors": metrics["total_errors"],
        "ro_errors": metrics["ro_errors"],
        "f1_electronic": metrics["f1_electronic"],
        "fold_wins": sum(value > 0 for value in fold_gains),
        "largest_positive_fold_gain_share": largest_share,
        "adopted": adopted,
    }])
    write_csv_atomic(aggregate, ARTIFACT_DIR / "assessment_aggregate.csv")
    error_analysis = {
        "scope": "official raw train OOF only",
        "metrics": metrics,
        "fold_gains_against_matching_v10": fold_gains,
        "largest_positive_fold_gain_share": largest_share,
        "recyclable_organic_error_rows": len(ro_errors),
        "manual_rules_or_overrides": False,
    }
    write_json_atomic(signed_document(error_analysis), ARTIFACT_DIR / "error_analysis.json")
    selection = signed_document({
        "status": "candidate_adopted" if adopted else "candidate_not_adopted",
        "adopted": adopted,
        "selection_basis": "signed train-only five-fold OOF",
        "oof_metrics": metrics,
        "v10_reference": V10_OOF_REFERENCE,
        "fold_gains_against_matching_v10": fold_gains,
        "largest_positive_fold_gain_share": largest_share,
        "thresholds": ADOPTION_THRESHOLDS,
        "checks": checks,
        "locked_recipe_signature": locked_recipe["signature"],
        "runtime_recipe_signature": locked_recipe["runtime_recipe_signature"],
        "test_access_state_at_signature": dict(TEST_ACCESS_STATE),
        "fallback_submission": False,
    })
    write_json_atomic(selection, ARTIFACT_DIR / "selection_decision.json")
    plot_confusion(metrics, FIGURES_DIR / "oof_confusion.png", "V12 train-only OOF")
    return selection, metrics


## 17. Conditional post-lock test discovery and five-fold inference

These functions do nothing when defined. The sole call is inside the worker after the signed adoption document is persisted and revalidated. Exactly one official 1,458-image test directory and one exact two-column template are accepted. Each fold adapter/head is loaded independently; only five-fold probability averaging and direct argmax are allowed.

In [13]:
def discover_test_and_template_after_adoption():
    persisted_selection = read_json(ARTIFACT_DIR / "selection_decision.json")
    persisted_lock = read_json(ARTIFACT_DIR / "locked_recipe_before_assessment.json")
    signed_adoption_checks = {
        "selection_signature": verify_signed_document(persisted_selection),
        "adopted": persisted_selection.get("adopted") is True,
        "all_train_only_checks": all(persisted_selection.get("checks", {}).values()),
        "recipe_lock_signature": verify_signed_document(persisted_lock),
        "recipe_match": (
            persisted_selection.get("runtime_recipe_signature")
            == persisted_lock.get("runtime_recipe_signature")
        ),
        "test_was_locked_at_selection": (
            persisted_selection.get("test_access_state_at_signature", {}).get(
                "test_images_accessed"
            ) is False
            and persisted_selection.get("test_access_state_at_signature", {}).get(
                "test_template_accessed"
            ) is False
        ),
        "worker_unlock_after_revalidation": TEST_ACCESS_STATE["unlocked"],
    }
    if not all(signed_adoption_checks.values()):
        raise RuntimeError(
            "Test discovery attempted without a fully revalidated signed adoption: {}".format(
                signed_adoption_checks
            )
        )
    test_candidates = []
    for current, directories, files in os.walk(KAGGLE_INPUT):
        path = Path(current)
        depth = len(path.relative_to(KAGGLE_INPUT).parts)
        directories[:] = [
            name for name in directories if not name.startswith(".") and depth < 5
        ]
        if path.name.lower() == "test":
            image_files = [
                path / name for name in files
                if Path(name).suffix.lower() in ALLOWED_IMAGE_SUFFIXES
            ]
            if len(image_files) == EXPECTED_TEST_ROWS:
                test_candidates.append((path.resolve(), image_files))
            directories[:] = []
    unique_test = {candidate[0]: candidate[1] for candidate in test_candidates}
    if len(unique_test) != 1:
        raise RuntimeError(
            "Expected exactly one 1,458-image official test directory, found {}".format(
                sorted(map(str, unique_test))
            )
        )
    test_root, image_files = next(iter(unique_test.items()))
    if any(path.is_symlink() or not path.is_file() for path in image_files):
        raise RuntimeError("Official test contains an invalid entry.")

    template_candidates = []
    for csv_path in KAGGLE_INPUT.rglob("*.csv"):
        try:
            frame = pd.read_csv(csv_path, dtype={"id": str})
        except Exception:
            continue
        if (
            tuple(frame.columns) == SUBMISSION_COLUMNS
            and len(frame) == EXPECTED_TEST_ROWS
            and frame["id"].notna().all()
            and frame["id"].is_unique
        ):
            template_candidates.append((csv_path.resolve(), frame))
    if len(template_candidates) != 1:
        raise RuntimeError(
            "Expected exactly one official 1,458-row submission template, found {}".format(
                [str(path) for path, _ in template_candidates]
            )
        )
    template_path, template = template_candidates[0]

    by_name = {}
    by_stem = {}
    for path in image_files:
        if path.name in by_name or path.stem in by_stem:
            raise RuntimeError("Test filename/stem collision.")
        by_name[path.name] = path
        by_stem[path.stem] = path
    ordered_paths = []
    for raw_id in template["id"].astype(str):
        candidate = by_name.get(raw_id)
        if candidate is None:
            candidate = by_stem.get(Path(raw_id).stem)
        if candidate is None:
            raise RuntimeError("Template ID has no matching test image: {}".format(raw_id))
        ordered_paths.append(str(candidate.resolve()))
    if len(set(ordered_paths)) != EXPECTED_TEST_ROWS:
        raise RuntimeError("Template IDs do not map one-to-one to test images.")

    TEST_ACCESS_STATE["test_images_accessed"] = True
    TEST_ACCESS_STATE["test_template_accessed"] = True
    return pd.DataFrame({
        "image_id": template["id"].astype(str).tolist(),
        "path": ordered_paths,
    }), template, test_root, template_path

class TestRows(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = self.transform(load_rgb_image(row["path"]))
        return {
            "pixel_values": image,
            "row_index": int(index),
            "image_id": str(row["image_id"]),
        }

def gather_test_probabilities(model, loader, test_frame, accelerator):
    model.eval()
    local_indices, local_ids, local_probabilities = [], [], []
    with torch.no_grad():
        for batch in loader:
            pixels = batch["pixel_values"].to(accelerator.device, non_blocking=True)
            with accelerator.autocast():
                logits = model(pixels)["main_logits"]
            local_probabilities.append(logits.float().softmax(dim=1).cpu().numpy())
            local_indices.extend(batch["row_index"].numpy().astype(np.int64).tolist())
            local_ids.extend([str(value) for value in batch["image_id"]])
    payload = {
        "indices": local_indices,
        "ids": local_ids,
        "probabilities": np.concatenate(local_probabilities, axis=0),
    }
    gathered = [None] * accelerator.num_processes
    dist.all_gather_object(gathered, payload)
    if not accelerator.is_main_process:
        return None
    indices = np.asarray([value for part in gathered for value in part["indices"]])
    ids = [value for part in gathered for value in part["ids"]]
    probabilities = np.concatenate([part["probabilities"] for part in gathered], axis=0)
    order = np.argsort(indices, kind="stable")
    indices, probabilities = indices[order], probabilities[order]
    ids = [ids[position] for position in order]
    checks = {
        "indices": np.array_equal(indices, np.arange(EXPECTED_TEST_ROWS)),
        "unique_indices": len(np.unique(indices)) == EXPECTED_TEST_ROWS,
        "ids": ids == test_frame["image_id"].astype(str).tolist(),
        "finite": np.isfinite(probabilities).all(),
    }
    if not all(checks.values()):
        raise RuntimeError("Distributed test gather failed: {}".format(checks))
    return normalize_probabilities(probabilities)

def infer_all_folds_once(test_frame, locked_runtime_signature, accelerator):
    fold_probabilities = []
    for fold in range(N_FOLDS):
        set_seed(SEED + 100 * fold, device_specific=False)
        model, processor, runtime_recipe = build_runtime_model(accelerator)
        if runtime_recipe.get("runtime_recipe_signature") is None:
            runtime_recipe = {
                **runtime_recipe,
                "augmentation": BASE_CONFIG["augmentation"],
                "loss": {
                    "main": "class-weighted CE", "class_weight_formula": "total/(3*class_count)",
                    "label_smoothing": LABEL_SMOOTHING,
                    "auxiliary": "balanced binary focal on class 0/2 only",
                    "focal_gamma": FOCAL_GAMMA, "auxiliary_weight": AUXILIARY_WEIGHT,
                    "auxiliary_inference": False,
                },
                "optimizer_groups": read_json(
                    ARTIFACT_DIR / "locked_recipe_before_assessment.json"
                )["runtime_recipe"]["optimizer_groups"],
                "scheduler": {"name": "cosine", "warmup_ratio": WARMUP_RATIO},
                "batching": {
                    "processes": 2, "microbatch_per_gpu": MICROBATCH_PER_GPU,
                    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
                    "effective_batch": EFFECTIVE_BATCH,
                },
                "seeds": {"global": SEED, "fold_formula": "42 + 100 * fold"},
                "epochs": {"maximum": MAX_EPOCHS, "early_stopping_patience": EARLY_STOPPING_PATIENCE},
                "selection_order": [
                    "higher Macro F1", "fewer errors", "fewer R/O errors", "lower normalized log loss"
                ],
                "development_thresholds": DEVELOPMENT_THRESHOLDS,
                "adoption_thresholds": ADOPTION_THRESHOLDS,
                "fold_assignment_signature": FOLD_ASSIGNMENT_SIGNATURE,
            }
            runtime_recipe["runtime_recipe_signature"] = sha256_json(runtime_recipe)
        if runtime_recipe["runtime_recipe_signature"] != locked_runtime_signature:
            raise RuntimeError("Test-time model recipe differs from the signed training recipe.")
        model.to(accelerator.device)
        load_fold_weights(model, fold, locked_runtime_signature)
        _, deterministic_transform = make_transforms(processor)
        dataset = TestRows(test_frame, deterministic_transform)
        sampler = ExactDistributedEvalSampler(
            len(dataset), accelerator.process_index, accelerator.num_processes
        )
        loader = DataLoader(
            dataset, batch_size=MICROBATCH_PER_GPU, sampler=sampler,
            shuffle=False, num_workers=LOADER_WORKERS_PER_PROCESS,
            pin_memory=True, persistent_workers=False,
            worker_init_fn=seed_loader_worker,
        )
        probabilities = gather_test_probabilities(
            model, loader, test_frame, accelerator
        )
        if accelerator.is_main_process:
            fold_probabilities.append(probabilities)
            print("Test inference fold {}/5 complete.".format(fold + 1), flush=True)
        accelerator.wait_for_everyone()
        del model, processor, dataset, sampler, loader
        gc.collect()
        torch.cuda.empty_cache()
    if accelerator.is_main_process:
        average = normalize_probabilities(np.mean(fold_probabilities, axis=0))
    else:
        average = None
    return average


## 18. Checkpoint reproduction and submission validation

Adopted test inference is repeated from newly constructed models and persisted adapters/heads. The averaged tensor must reproduce within 1e-6 with identical argmax. Submission validation is exact and no post-processing path exists.

In [14]:
def run_adopted_test(selection, locked_recipe, accelerator):
    if not selection["adopted"] or not verify_signed_document(selection):
        raise RuntimeError("Unsigned or negative selection cannot unlock test.")
    if accelerator.is_main_process:
        test_frame, template, test_root, template_path = (
            discover_test_and_template_after_adoption()
        )
        payload = {
            "records": test_frame.to_dict("records"),
            "template_records": template.to_dict("records"),
            "test_root": str(test_root),
            "template_path": str(template_path),
        }
    else:
        payload = None
    payload = broadcast_object(accelerator, payload)
    test_frame = pd.DataFrame(payload["records"])
    template = pd.DataFrame(payload["template_records"], columns=SUBMISSION_COLUMNS)
    TEST_ACCESS_STATE["test_images_accessed"] = True
    TEST_ACCESS_STATE["test_template_accessed"] = True

    runtime_signature = locked_recipe["runtime_recipe_signature"]
    first = infer_all_folds_once(test_frame, runtime_signature, accelerator)
    accelerator.wait_for_everyone()
    reproduced = infer_all_folds_once(test_frame, runtime_signature, accelerator)
    if accelerator.is_main_process:
        difference = float(np.max(np.abs(first - reproduced)))
        predictions_identical = np.array_equal(
            first.argmax(axis=1), reproduced.argmax(axis=1)
        )
        reproduction_passed = difference <= RELOAD_TOLERANCE and predictions_identical
        if not reproduction_passed:
            raise RuntimeError("Five-fold test inference reproduction failed.")

        write_npy_atomic(first, ARTIFACT_DIR / "test_probabilities.npy")
        predictions = first.argmax(axis=1).astype(np.int64)
        submission = pd.DataFrame({
            "id": template["id"].astype(str).to_numpy(),
            "predicted": predictions,
        })
        checks = {
            "row_count_1458": len(submission) == EXPECTED_TEST_ROWS,
            "exact_columns_and_order": tuple(submission.columns) == SUBMISSION_COLUMNS,
            "unique_ids": submission["id"].is_unique,
            "no_missing": not submission.isna().any().any(),
            "labels_0_1_2": submission["predicted"].isin([0, 1, 2]).all(),
            "template_order_exact": submission["id"].tolist() == template["id"].astype(str).tolist(),
            "probabilities_finite": np.isfinite(first).all(),
            "probability_rows_normalized": np.allclose(first.sum(axis=1), 1.0, atol=1e-6),
        }
        if not all(checks.values()):
            raise RuntimeError("Submission contract failed: {}".format(checks))
        submission_path = ARTIFACT_DIR / "submission_{}.csv".format(
            safe_team_name(TEAM_NAME)
        )
        write_csv_atomic(submission, submission_path)
        manifest = signed_document({
            "selection_signature": selection["signature"],
            "locked_recipe_signature": locked_recipe["signature"],
            "test_root": payload["test_root"],
            "template_path": payload["template_path"],
            "test_rows": EXPECTED_TEST_ROWS,
            "view": "deterministic original 320x320",
            "fold_checkpoints_loaded_independently": 5,
            "probability_operation": "arithmetic mean of five fold probabilities",
            "tta": False,
            "seed_ensemble": False,
            "external_blend": False,
            "logit_shift": False,
            "threshold_tuning": False,
            "object_rules": False,
            "row_corrections": False,
            "max_reload_probability_difference": difference,
            "reload_tolerance": RELOAD_TOLERANCE,
            "identical_argmax": predictions_identical,
            "submission_checks": checks,
            "predicted_class_distribution": {
                str(label): int((predictions == label).sum()) for label in range(NUM_CLASSES)
            },
            "submission": submission_path.name,
        })
        write_json_atomic(manifest, ARTIFACT_DIR / "test_inference_manifest.json")
        reproduction = read_json(ARTIFACT_DIR / "checkpoint_reproduction_check.json")
        unsigned = {key: value for key, value in reproduction.items() if key != "signature"}
        unsigned["test_five_fold_reload"] = {
            "max_absolute_probability_difference": difference,
            "tolerance": RELOAD_TOLERANCE,
            "identical_argmax": predictions_identical,
            "passed": reproduction_passed,
        }
        write_json_atomic(signed_document(unsigned), ARTIFACT_DIR / "checkpoint_reproduction_check.json")
        result = {
            "submission": submission_path.name,
            "predicted_class_distribution": manifest["predicted_class_distribution"],
            "test_reproduction_max_difference": difference,
        }
    else:
        result = None
    return broadcast_object(accelerator, result)


## 19. Distributed orchestration, final summary, figures, and artifact bundle

One notebook_launcher call owns the entire state machine. Fold-0 rejection and assessment rejection return cleanly before the sole post-lock test call. Only rank zero writes distributed artifacts.

In [15]:
def finalize_rank_zero(status, development_decision, selection, fold_metrics, oof_metrics=None, test_result=None):
    summary = {
        "status": status,
        "experiment": EXPERIMENT_NAME,
        "model_id": MODEL_ID,
        "model_revision": RESOLVED_MODEL_REVISION,
        "fold_signature": FOLD_ASSIGNMENT_SIGNATURE,
        "development_decision": development_decision["status"],
        "development_checks": development_decision["checks"],
        "selection": None if selection is None else selection["status"],
        "candidate_adopted": bool(selection and selection.get("adopted")),
        "fold_metrics": {str(key): value for key, value in fold_metrics.items()},
        "oof_metrics": oof_metrics,
        "test_result": test_result,
        "test_access_state": dict(TEST_ACCESS_STATE),
        "fallback_v10_submission": False,
        "external_data_or_labels_used": False,
        "tta_blend_threshold_or_row_override": False,
        "runtime_seconds": time.perf_counter() - NOTEBOOK_STARTED_AT,
        "artifact_bundle": "artifact_bundle.zip",
    }
    write_json_atomic(signed_document(summary), ARTIFACT_DIR / "final_run_summary.json")
    required = [
        "config.json", "environment.json", "data_contract.json",
        "fold_assignments.csv", "fold_signature.json", "model_manifest.json",
        "dependency_audit.json", "lora_target_audit.json", "preflight_report.json",
        "development_decision.json", "error_analysis.json", "selection_decision.json",
        "checkpoint_reproduction_check.json", "final_run_summary.json",
    ]
    if development_decision["all_checks_passed"]:
        required.extend([
            "locked_recipe_before_assessment.json", "assessment_per_fold.csv",
            "assessment_aggregate.csv", "oof_probabilities.npy",
            "oof_predictions.csv", "oof_metrics.json",
            "oof_recyclable_organic_errors.csv",
        ])
    if selection and selection.get("adopted"):
        required.extend([
            "test_probabilities.npy", "test_inference_manifest.json",
            "submission_{}.csv".format(safe_team_name(TEAM_NAME)),
        ])
    missing = [name for name in required if not (ARTIFACT_DIR / name).is_file()]
    if missing:
        raise RuntimeError("Required final artifacts are missing: {}".format(missing))
    for fold in fold_metrics:
        if not fold_paths(fold)["complete"].is_file():
            raise RuntimeError("Fold {} completion artifact is missing.".format(fold))
    bundle = build_artifact_bundle()
    print("Final status:", status)
    print("Artifacts:", ARTIFACT_DIR)
    print("Bundle:", bundle)

def distributed_v12_run():
    ddp_kwargs = DistributedDataParallelKwargs(
        broadcast_buffers=False, find_unused_parameters=False
    )
    accelerator = Accelerator(
        mixed_precision="fp16",
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        kwargs_handlers=[ddp_kwargs],
    )
    if accelerator.num_processes != 2 or accelerator.device.type != "cuda":
        raise RuntimeError("V12 requires exactly two CUDA worker processes.")
    set_seed(SEED, device_specific=False)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.set_float32_matmul_precision("high")

    properties = torch.cuda.get_device_properties(accelerator.device)
    local_gpu = {
        "process_index": accelerator.process_index,
        "local_process_index": accelerator.local_process_index,
        "device": str(accelerator.device),
        "name": properties.name,
        "total_memory_gib": properties.total_memory / (2 ** 30),
    }
    all_gpus = [None] * accelerator.num_processes
    dist.all_gather_object(all_gpus, local_gpu)
    gpu_contract = (
        len(all_gpus) == 2
        and len({item["device"] for item in all_gpus}) == 2
        and all("T4" in item["name"].upper() for item in all_gpus)
    )
    if not gpu_contract:
        raise RuntimeError("Expected one Tesla T4 per process: {}".format(all_gpus))
    if accelerator.is_main_process:
        environment = read_json(ARTIFACT_DIR / "environment.json")
        environment.update({
            "cuda_runtime": torch.version.cuda,
            "cudnn": torch.backends.cudnn.version(),
            "gpus": all_gpus,
            "mixed_precision": "fp16",
            "distributed_backend": dist.get_backend(),
            "world_size": accelerator.num_processes,
        })
        write_json_atomic(environment, ARTIFACT_DIR / "environment.json")
    accelerator.wait_for_everyone()

    fold_metrics = {}
    fold_metrics[0], runtime_recipe = run_fold(0, accelerator, preflight=True)
    if accelerator.is_main_process:
        development = make_development_decision(fold_metrics[0], runtime_recipe)
        write_json_atomic(development, ARTIFACT_DIR / "development_decision.json")
        plot_confusion(
            fold_metrics[0], FIGURES_DIR / "fold0_confusion.png",
            "V12 fold 0 development",
        )
        error_analysis = signed_document({
            "scope": "official raw train fold 0 only",
            "metrics": fold_metrics[0],
            "v10_reference": V10_FOLD_REFERENCE[0],
            "manual_rules_or_overrides": False,
        })
        write_json_atomic(error_analysis, ARTIFACT_DIR / "error_analysis.json")
    else:
        development = None
    development = broadcast_object(accelerator, development)
    if not verify_signed_document(development):
        raise RuntimeError("Development decision signature failed.")

    if not development["all_checks_passed"]:
        if accelerator.is_main_process:
            selection = signed_document({
                "status": "candidate_rejected",
                "adopted": False,
                "selection_basis": "fold-0 train-only gate",
                "development_decision_signature": development["signature"],
                "assessment_folds_run": [],
                "test_accessed": False,
                "fallback_submission": False,
            })
            write_json_atomic(selection, ARTIFACT_DIR / "selection_decision.json")
            finalize_rank_zero(
                "candidate_rejected", development, selection, fold_metrics
            )
        accelerator.wait_for_everyone()
        return

    if accelerator.is_main_process:
        locked_recipe = persist_recipe_lock(runtime_recipe, development)
    else:
        locked_recipe = None
    locked_recipe = broadcast_object(accelerator, locked_recipe)
    if (
        not verify_signed_document(locked_recipe)
        or locked_recipe["runtime_recipe_signature"]
        != runtime_recipe["runtime_recipe_signature"]
    ):
        raise RuntimeError("Signed recipe lock failed before assessment.")

    for fold in ASSESSMENT_FOLDS:
        fold_metrics[fold], fold_recipe = run_fold(fold, accelerator, preflight=False)
        if (
            fold_recipe["runtime_recipe_signature"]
            != locked_recipe["runtime_recipe_signature"]
        ):
            raise RuntimeError("Assessment fold recipe drift detected.")
        accelerator.wait_for_everyone()

    if accelerator.is_main_process:
        selection, oof_metrics = build_oof_and_selection(
            fold_metrics, locked_recipe
        )
    else:
        selection, oof_metrics = None, None
    selection = broadcast_object(accelerator, selection)
    oof_metrics = broadcast_object(accelerator, oof_metrics)
    if not verify_signed_document(selection):
        raise RuntimeError("Persisted train-only selection signature failed.")

    if not selection["adopted"]:
        if accelerator.is_main_process:
            finalize_rank_zero(
                "candidate_not_adopted", development, selection,
                fold_metrics, oof_metrics=oof_metrics,
            )
        accelerator.wait_for_everyone()
        return

    # Sole test unlock: re-read and validate the fully persisted train-only adoption.
    if accelerator.is_main_process:
        persisted_selection = read_json(ARTIFACT_DIR / "selection_decision.json")
        persisted_lock = read_json(ARTIFACT_DIR / "locked_recipe_before_assessment.json")
        full_adoption_revalidated = (
            verify_signed_document(persisted_selection)
            and persisted_selection["adopted"] is True
            and all(persisted_selection["checks"].values())
            and verify_signed_document(persisted_lock)
            and persisted_selection["runtime_recipe_signature"]
            == persisted_lock["runtime_recipe_signature"]
            and persisted_selection["test_access_state_at_signature"]["test_images_accessed"] is False
            and persisted_selection["test_access_state_at_signature"]["test_template_accessed"] is False
        )
    else:
        full_adoption_revalidated = None
    full_adoption_revalidated = broadcast_object(
        accelerator, full_adoption_revalidated
    )
    if not full_adoption_revalidated:
        raise RuntimeError("Full signed adoption did not revalidate; test remains locked.")

    TEST_ACCESS_STATE["unlocked"] = True
    test_result = run_adopted_test(selection, locked_recipe, accelerator)
    if accelerator.is_main_process:
        finalize_rank_zero(
            "candidate_adopted_complete", development, selection,
            fold_metrics, oof_metrics=oof_metrics, test_result=test_result,
        )
    accelerator.wait_for_everyone()


## 20. Completion contract and launch

Run All invokes exactly one notebook-safe two-process launcher. The parent has made zero CUDA calls. ACCELERATE_DEBUG_MODE is enabled so a contaminated notebook session fails before launch. The final cell reports the persisted outcome; a rejected candidate is a successful, submission-free completion.

In [16]:
PARENT_CUDA_AUDIT = {
    "pytorch_nvml_based_cuda_check": os.environ.get("PYTORCH_NVML_BASED_CUDA_CHECK"),
    "accelerate_debug_mode": os.environ.get("ACCELERATE_DEBUG_MODE"),
    "cuda_initialized_before_launcher": bool(torch.cuda.is_initialized()),
}
print("Parent CUDA audit:", PARENT_CUDA_AUDIT)
if PARENT_CUDA_AUDIT["cuda_initialized_before_launcher"]:
    raise RuntimeError(
        "CUDA was initialized in the parent process before notebook_launcher. "
        "Start a fresh Kaggle Save & Run All session with this notebook."
    )

notebook_launcher(
    distributed_v12_run,
    args=(),
    num_processes=2,
    mixed_precision="fp16",
)

FINAL_SUMMARY_PATH = ARTIFACT_DIR / "final_run_summary.json"
if not FINAL_SUMMARY_PATH.is_file():
    raise RuntimeError("Distributed run returned without a final summary.")
FINAL_SUMMARY = read_json(FINAL_SUMMARY_PATH)
if not verify_signed_document(FINAL_SUMMARY):
    raise RuntimeError("Final summary signature is invalid.")
print(json.dumps({
    "status": FINAL_SUMMARY["status"],
    "candidate_adopted": FINAL_SUMMARY["candidate_adopted"],
    "test_access_state": FINAL_SUMMARY["test_access_state"],
    "artifact_directory": str(ARTIFACT_DIR),
    "artifact_bundle": str(ARTIFACT_DIR / "artifact_bundle.zip"),
}, indent=2))


Parent CUDA audit: {'pytorch_nvml_based_cuda_check': '1', 'accelerate_debug_mode': '1', 'cuda_initialized_before_launcher': False}
Launching training on 2 CUDAs.


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:897.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:897.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


fold=0 epoch=1 batch=100/5306 elapsed_min=0.7
fold=0 epoch=1 batch=200/5306 elapsed_min=1.3
fold=0 epoch=1 batch=300/5306 elapsed_min=1.9
fold=0 epoch=1 batch=400/5306 elapsed_min=2.6
fold=0 epoch=1 batch=500/5306 elapsed_min=3.2
fold=0 epoch=1 batch=600/5306 elapsed_min=3.8
fold=0 epoch=1 batch=700/5306 elapsed_min=4.5
fold=0 epoch=1 batch=800/5306 elapsed_min=5.1
fold=0 epoch=1 batch=900/5306 elapsed_min=5.7
fold=0 epoch=1 batch=1000/5306 elapsed_min=6.4
fold=0 epoch=1 batch=1100/5306 elapsed_min=7.0
fold=0 epoch=1 batch=1200/5306 elapsed_min=7.6
fold=0 epoch=1 batch=1300/5306 elapsed_min=8.2
fold=0 epoch=1 batch=1400/5306 elapsed_min=8.8
fold=0 epoch=1 batch=1500/5306 elapsed_min=9.4
fold=0 epoch=1 batch=1600/5306 elapsed_min=9.9
fold=0 epoch=1 batch=1700/5306 elapsed_min=10.5
fold=0 epoch=1 batch=1800/5306 elapsed_min=11.1
fold=0 epoch=1 batch=1900/5306 elapsed_min=11.7
fold=0 epoch=1 batch=2000/5306 elapsed_min=12.3
fold=0 epoch=1 batch=2100/5306 elapsed_min=12.8
fold=0 epoch=1 ba

W0728 19:28:53.943000 23 torch/multiprocessing/spawn.py:165] Terminating process 79 via signal SIGTERM
E0728 19:28:54.042000 23 torch/distributed/elastic/multiprocessing/api.py:833] failed (exitcode: 1) local_rank: 0 (pid: 78) of fn: distributed_v12_run (start_method: fork)
E0728 19:28:54.042000 23 torch/distributed/elastic/multiprocessing/api.py:833] Traceback (most recent call last):
E0728 19:28:54.042000 23 torch/distributed/elastic/multiprocessing/api.py:833]   File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/api.py", line 788, in _poll
E0728 19:28:54.042000 23 torch/distributed/elastic/multiprocessing/api.py:833]     self._pc.join(-1)
E0728 19:28:54.042000 23 torch/distributed/elastic/multiprocessing/api.py:833]   File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/spawn.py", line 211, in join
E0728 19:28:54.042000 23 torch/distributed/elastic/multiprocessing/api.py:833]     raise ProcessRaisedException(msg, error_index, faile

ChildFailedError: 
============================================================
distributed_v12_run FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-07-28_19:28:53
  host      : e2dd156b1813
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 78)
  error_file: /tmp/torchelastic_jjs2dxwa/none_zeme0k0p/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper
      return f(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^
    File "/tmp/ipykernel_23/1212516650.py", line 101, in distributed_v12_run
      fold_metrics[0], runtime_recipe = run_fold(0, accelerator, preflight=True)
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "/tmp